In [0]:
* ================================================================
* Chapter 3 Analysis: School Culture and Pupil Progress
* 3-stage OLS on 102 gold-standard (visit + interview) schools
* Outcome: P8 component 2-year averages (primary) + robustness
* HC3 robust standard errors throughout
* ================================================================

* Ensure user ado packages (esttab/estout) are on the search path
adopath ++ "C:\Users\damia\ado\plus"

* Confirm esttab is findable
which esttab

  [1]              "C:\Users\damia\ado\plus"
  [2]  (BASE)      "C:\Program Files\StataNow19/ado\base/"
  [3]  (SITE)      "C:\Program Files\StataNow19/ado\site/"
  [4]              "."
  [5]  (PERSONAL)  "C:\Users\damia\ado\personal/"
  [6]  (PLUS)      "C:\Users\damia\ado\plus/"
  [7]  (OLDPLACE)  "c:\ado/"
C:\Users\damia\ado\plus\e\esttab.ado
*! version 2.1.4  13apr2026  Ben Jann
*! wrapper for estout


In [1]:
* ---- 1. Load data ----
import delimited "C:/Users/damia/OneDrive/Documents/Schools Project/analysis_dataset.csv", ///
    clear stringcols(_all) case(lower)

* Strip % sign from eal and sen (stored as "57.70%" strings)
foreach v of varlist eal sen {
    replace `v' = subinstr(`v', "%", "", .)
    destring `v', replace
}

* Destring all numeric analysis columns
* NO COMPOSITE (5 Aug 2026). gs_*_composite and gs_*_score_v1 are gone: a 60/40 blend of an
* enacted and an espoused measure is not a measure of either, and the two agree at only
* +0.243 / +0.190 / +0.184. The gold standard is now two named columns per construct.
* gs_*_visit is retained as an alias of gs_*_enacted, so every tier1 model below is
* unchanged -- they were already reading the visit score, not the composite.
local numvars gs_warmth_visit gs_strictness_visit gs_teaching_visit ///
    gs_warmth_enacted gs_strictness_enacted gs_teaching_enacted ///
    gs_warmth_espoused gs_strictness_espoused gs_teaching_espoused ///
    gs_w1 gs_w2 gs_w3_adj gs_s1 gs_s2 gs_s3 gs_s4 gs_t1 gs_t2 ///
    p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg ///
    p8mea_2324 p8meaeng_2324 p8meamat_2324 p8meaebac_2324 p8meaopen_2324 ///
    att8screng_2425 att8scrmat_2425 att8screbac_2425 att8scropen_2425 ///
    ks2 fsm log_size academy urban_bin selective ///
    years_since_ofsted ofsted_grade_2019 ///
    ofsted_llmstrictnessscore ///
    /// Segmented interview scorers (5 Aug 2026), NOT the v6 bundled ones. v6's scale
    /// collapsed and its dimensions leaked (S x M = +0.804); these are four separate
    /// calls over disjoint text. Names changed so the swap could not be silent.
    trx_warmth trx_strictness trx_management trx_teaching ///
    semh_baseline_2016 semh_current size
destring `numvars', replace force

* Sample flags
* tier1 = ENACTED culture, the schools that were visited. 103 since the URN-join fix
* recovered Trinity CofE (URN 136538); the thesis text still says 102 in places.
* tier2 = ESPOUSED culture, every school with a headteacher interview. This was
* previously written !missing(gs_warmth_composite), which selected exactly these rows --
* the composite existed for all 304 interviewed schools, being 100% espoused for the 201
* that were never visited. Same sample, honest name.
gen tier1 = !missing(gs_warmth_enacted)    // 103 visited schools
gen tier2 = !missing(gs_warmth_espoused)   // 304 interview schools

count if tier1
count if tier2

(encoding automatically selected: UTF-8)


(104 vars, 3,332 obs)
(3,268 real changes made)


eal: all characters numeric; replaced as double
(64 missing values generated)


(3,268 real changes made)


sen: all characters numeric; replaced as double
(64 missing values generated)


...

trx_warmth: all characters numeric; replaced as byte
(3042 missing values generated)


trx_strictness: all characters numeric; replaced as byte
(3042 missing values generated)
trx_management: all characters numeric; replaced as byte
(3042 missing values generated)


trx_teaching: all characters numeric; replaced as byte
(3042 missing values generated)
semh_baseline_2016: contains nonnumeric characters; replaced as int
(325 missing values generated)


semh_current: all characters numeric; replaced as int


size: all characters numeric; replaced as int
(6 missing values generated)
  103
  304


In [2]:
* ---- 2. Define global macros ----

* Base controls (continuous)
global ctrl_cont "ks2 fsm eal sen log_size years_since_ofsted"

* Binary school-type controls
global ctrl_bin "academy urban_bin selective"

* Pre-COVID Ofsted grade dummies (base = Outstanding, grade 1)
* Missing grade (7 schools) are dropped from regressions with this macro
global ctrl_ofsted "2.ofsted_grade_2019 3.ofsted_grade_2019 4.ofsted_grade_2019"

* Full control set (primary spec)
global controls "$ctrl_cont $ctrl_bin $ctrl_ofsted"

* Controls without Ofsted grade (sensitivity — keeps all 102 schools)
global controls_ngrade "$ctrl_cont $ctrl_bin"

* Primary outcome variables: overall P8 first, then 4 components
global outcomes "p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg"

* Culture predictors
global W  "gs_warmth_visit"
global S  "gs_strictness_visit"
global T  "gs_teaching_visit"
global WS "gs_warmth_visit gs_strictness_visit"

display "Macros defined."

Macros defined.


In [3]:
* ---- 3. Descriptive check on Tier 1 sample ----
preserve
keep if tier1

display _newline "=== Tier 1 descriptive statistics (N=102 visited schools) ==="
summarize $WS $T $outcomes ks2 fsm eal sen log_size academy urban_bin selective ///
    years_since_ofsted ofsted_grade_2019

display _newline "Pre-COVID Ofsted grade distribution:"
tabulate ofsted_grade_2019, missing

display _newline "Pairwise correlations (culture predictors and main outcomes):"
correlate $WS $T $outcomes

restore

(Note: Below code run with echo to enable preserve/restore functionality.)




(3,229 observations deleted)


=== Tier 1 descriptive statistics (N=102 visited schools) ===


    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
gs_warmth_~t |        103    6.558835    .9882281       4.33       8.81
gs_strictn~t |        103    7.016117    .8687756       4.85       8.96
gs_teachin~t |        103    6.967476     .833038       4.98       9.02
   p8mea_avg |        103    .2515534    .4777429        -.7      1.345
p8meaeng_avg |        103    .2563592    .5071461       -.75      1.545
-------------+---------------------------------------------------------
p8meamat_avg |        103    .2698544    .4494602      -.595      1.345
p8meaebac_~g |        103    .2972816    .5486453       -.79       1.49
p8meaopen_~g |        103    .1717476     .527677     -1.095      1.435
         ks2 |        103    104.9087    2.895335       99.8        117
         fsm |        103    29.21262  

In [4]:
* ================================================================
* PRIMARY REGRESSIONS — TIER 1 (n ≈ 95, dropped 7 with no pre-COVID grade)
* Stage 1: Total culture effect  y = α + β₁W + β₂S + X'γ + ε
* ================================================================
estimates clear

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    regress `outcome' $WS $controls if tier1, vce(hc3)
    estimates store s1_`lbl'
    display _newline "Stage 1 — `lbl' (W+S, n=" e(N) "):"
    display "  β_W = " %6.3f _b[gs_warmth_visit] ///
            "  (se=" %6.3f _se[gs_warmth_visit] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_warmth_visit]/_se[gs_warmth_visit])))
    display "  β_S = " %6.3f _b[gs_strictness_visit] ///
            "  (se=" %6.3f _se[gs_strictness_visit] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_strictness_visit]/_se[gs_strictness_visit])))
    display "  R² = " %6.4f e(r2)
}


Linear regression                               Number of obs     =         96
                                                F(13, 82)         =      15.48
                                                Prob > F          =     0.0000
                                                R-squared         =     0.6501
                                                Root MSE          =     .30162



------------------------------------------------------------------------------


             |             Robust HC3


   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------


gs_warmth_~t |

...


Stage 1 — Overall (W+S, n=96):
 β_W =  0.150 (se= 0.045) p=0.001
 β_S =  0.130 (se= 0.047) p=0.007
 R² = 0.6501

Linear regression                               Number of obs     =         96
                                                F(12, 82)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.6382
                                                Root MSE          =     .32726



------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1765795    .043382     4.07   0.000     .0902788    .2628801
gs_strictn~t |   .0864345   .0493691     1.75   0.084    -.0117763    .1846454
         ks2 |   .1231733   .0299718     4.11   0.000     .0635499    .1827968
         fsm |   -.004286   .0044701    -0.96   0.340    -.0131784    .0046064
         eal |   .0140587    .002112     6.66   0.000     .0098573    .0182602
         sen |   .0065597   .0052383     1.25   0.214    -.0038609    .0169802
    log_size |  -.1292581   .1065153    -1.21   0.228    -.3411511    .0826348
years_sinc~d |  -.0084408   .0110925    -0.76   0.449    -.0305073    .0136257
     academy |   .0816477   .0992325     0.82   0.413    -.1157574    .2790528
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1176421   .0556669     2.11   0.038      .006903    .2283812
gs_strictn~t |   .1094054   .0520123     2.10   0.038     .0059364    .2128744
         ks2 |   .0759678   .0243386     3.12   0.002     .0275507    .1243849
         fsm |  -.0079994    .003861    -2.07   0.041    -.0156801   -.0003186
         eal |   .0114507   .0018001     6.36   0.000     .0078697    .0150318
         sen |   .0097373   .0053759     1.81   0.074     -.000957    .0204316
    log_size |   -.026307   .1182283    -0.22   0.824    -.2615007    .2088866
years_sinc~d |  -.0110893   .0109021    -1.02   0.312     -.032777    .0105984
     academy |   .0049594   .1051502     0.05   0.962    -.2042178    .2141366
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1645371   .0579617     2.84   0.006     .0492327    .2798414
gs_strictn~t |   .1690583   .0580863     2.91   0.005     .0535062    .2846104
         ks2 |    .115174   .0350296     3.29   0.001     .0454888    .1848591
         fsm |  -.0060866   .0045441    -1.34   0.184    -.0151262     .002953
         eal |   .0131453   .0021983     5.98   0.000     .0087723    .0175184
         sen |    .008246   .0064833     1.27   0.207    -.0046513    .0211433
    log_size |   -.067422   .1436475    -0.47   0.640    -.3531826    .2183386
years_sinc~d |  -.0093143   .0122223    -0.76   0.448    -.0336283    .0149998
     academy |   .0539755   .1130613     0.48   0.634    -.1709394    .2788903
   urban_bin |


Linear regression                               Number of obs     =         96
                                                F(13, 82)         =      12.44
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5653
                                                Root MSE          =     .35832

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1457066   .0586771     2.48   0.015     .0289791    .2624341
gs_strictn~t |    .135027   .0646678     2.09   0.040     .0063821    .2636718
         ks2 |   .1207491   .0284083     4.25   0.000      .064236    .1772622
         fsm |  -.0041004   .0048353    -0.85   0.399    -.0137194    .0055186
         eal

In [5]:
* ================================================================
* Stage 2: Teaching quality benchmark  y = α + γT + X'γ + ε
* ================================================================
foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    regress `outcome' $T $controls if tier1, vce(hc3)
    estimates store s2_`lbl'
    display _newline "Stage 2 — `lbl' (T only, n=" e(N) "):"
    display "  γ_T = " %6.3f _b[gs_teaching_visit] ///
            "  (se=" %6.3f _se[gs_teaching_visit] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_teaching_visit]/_se[gs_teaching_visit])))
    display "  R² = " %6.4f e(r2)
}


Linear regression                               Number of obs     =         96
                                                F(12, 83)         =      10.80
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5794
                                                Root MSE          =      .3287



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_teachin~t |   .2514004   .0521197     4.82   0.000     .1477365    .3550643
         ks2 |   .0948544   .0258762     3.67   0.000     .0433876    .1463212
         fsm |  -.0111634   .0043537    -2.56   0.012    -.0198228    -.002504
         eal |   .0134553   .0021224     6.34   0.000      .009234    .0176767
         sen |   .0069501   .0054081     1.29   0.202    -.0038064    .0177066
    log_size |  -.1028154   .1135367    -0.91   0.368    -.3286354    .1230046
years_sinc~d |   -.007834   .0105987    -0.74   0.462    -.0289144    .0132463
     academy |  -.0058248   .0982238    -0.06   0.953     -.201188    .1895384
   urban_bin |   .0698811   .1108045     0.63   0.530    -.1505047    .2902668
   selective |


Linear regression                               Number of obs     =         96
                                                F(11, 83)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5714
                                                Root MSE          =     .35403

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_teachin~t |   .2317333    .052731     4.39   0.000     .1268535    .3366131
         ks2 |   .1084585   .0302391     3.59   0.001      .048314    .1686029
         fsm |  -.0093091   .0054051    -1.72   0.089    -.0200596    .0014414
         eal |    .015593   .0025115     6.21   0.000     .0105978    .0205882
         sen


Linear regression                               Number of obs     =         96
                                                F(12, 83)         =       7.68
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4891
                                                Root MSE          =     .34846

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_teachin~t |   .2047091   .0598857     3.42   0.001     .0855988    .3238195
         ks2 |    .063697   .0230257     2.77   0.007     .0178998    .1094941
         fsm |  -.0126502   .0039015    -3.24   0.002      -.02041   -.0048904
         eal |   .0132844   .0019822     6.70   0.000     .0093419    .0172269
         sen


Linear regression                               Number of obs     =         96
                                                F(11, 83)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5223
                                                Root MSE          =      .4089

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_teachin~t |   .2811263     .06601     4.26   0.000     .1498351    .4124175
         ks2 |    .098131   .0336363     2.92   0.005     .0312297    .1650322
         fsm |  -.0124437   .0053502    -2.33   0.022    -.0230851   -.0018023
         eal |   .0156315   .0027579     5.67   0.000     .0101462    .0211169
         sen

             |
ofsted_~2019 |
          3  |  -.1291032    .159988    -0.81   0.422    -.4473128    .1891064
          4  |  -.2043498   .1451935    -1.41   0.163    -.4931338    .0844342
             |
       _cons |  -11.09355   3.727718    -2.98   0.004    -18.50783   -3.679265
------------------------------------------------------------------------------

Stage 2 — EBaC (T only, n=96):
 γ_T =  0.281 (se= 0.066) p=0.000
 R² = 0.5223

Linear regression                               Number of obs     =         96
                                                F(12, 83)         =       8.84
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5270
                                                Root MSE          =     .37153

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t  

  -.0338507   .1182472    -0.29   0.775    -.2690397    .2013382
years_sinc~d |  -.0096077   .0113729    -0.84   0.401    -.0322279    .0130124
     academy |  -.0302916   .1116198    -0.27   0.787    -.2522989    .1917157
   urban_bin |   .0786194   .1167177     0.67   0.502    -.1535274    .3107662
   selective |  -.8423344   .3110774    -2.71   0.008    -1.461055    -.223614
             |
ofsted_~2019 |
          3  |  -.0291761     .20123    -0.14   0.885    -.4294144    .3710622
          4  |  -.1327442   .1843576    -0.72   0.474    -.4994241    .2339357
             |
       _cons |  -12.44623   3.105489    -4.01   0.000    -18.62292   -6.269541
------------------------------------------------------------------------------

Stage 2 — Open (T only, n=96):
 γ_T =  0.274 (se= 0.058) p=0.000
 R² = 0.5270


In [6]:
* ================================================================
* Stage 3: Direct culture effect net of teaching  y = α + β₁W + β₂S + γT + X'γ + ε
* ================================================================
foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    regress `outcome' $WS $T $controls if tier1, vce(hc3)
    estimates store s3_`lbl'
    display _newline "Stage 3 — `lbl' (W+S+T, n=" e(N) "):"
    display "  β_W = " %6.3f _b[gs_warmth_visit] ///
            "  (se=" %6.3f _se[gs_warmth_visit] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_warmth_visit]/_se[gs_warmth_visit])))
    display "  β_S = " %6.3f _b[gs_strictness_visit] ///
            "  (se=" %6.3f _se[gs_strictness_visit] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_strictness_visit]/_se[gs_strictness_visit])))
    display "  γ_T = " %6.3f _b[gs_teaching_visit] ///
            "  (se=" %6.3f _se[gs_teaching_visit] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_teaching_visit]/_se[gs_teaching_visit])))
    display "  R² = " %6.4f e(r2)
}


Linear regression                               Number of obs     =         96
                                                F(14, 81)         =      14.33
                                                Prob > F          =     0.0000
                                                R-squared         =     0.6504
                                                Root MSE          =     .30335



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1407191   .0634007     2.22   0.029     .0145716    .2668666
gs_strictn~t |   .1258641   .0468155     2.69   0.009     .0327159    .2190124
gs_teachin~t |    .018776   .0815825     0.23   0.819    -.1435476    .1810996
         ks2 |   .1089801   .0271996     4.01   0.000     .0548614    .1630987
         fsm |  -.0059223   .0042754    -1.39   0.170     -.014429    .0025845
         eal |   .0114575   .0018757     6.11   0.000     .0077254    .0151896
         sen |   .0070354   .0050718     1.39   0.169     -.003056    .0171267
    log_size |  -.0434603   .1052647    -0.41   0.681    -.2529041    .1659835
years_sinc~d |  -.0100765    .010346    -0.97   0.333    -.0306618    .0105088
     academy |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1863763   .0644953     2.89   0.005     .0580509    .3147017
gs_strictn~t |   .0904832   .0499238     1.81   0.074    -.0088496     .189816
gs_teachin~t |  -.0192151   .0845625    -0.23   0.821    -.1874679    .1490378
         ks2 |   .1242554   .0317019     3.92   0.000     .0611786    .1873322
         fsm |   -.003838   .0051792    -0.74   0.461    -.0141429     .006467
         eal |   .0138728   .0021188     6.55   0.000     .0096572    .0180885
         sen |   .0065372   .0052786     1.24   0.219    -.0039656      .01704
    log_size |  -.1242265   .1064044    -1.17   0.246    -.3359378    .0874848
years_sinc~d |  -.0086861   .0114029    -0.76   0.448    -.0313743    .0140022
     academy |


Linear regression                               Number of obs     =         96
                                                F(13, 81)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5381
                                                Root MSE          =     .33537



------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1070335   .0704929     1.52   0.133    -.0332253    .2472924
gs_strictn~t |   .1050213   .0579225     1.81   0.074    -.0102264    .2202689
gs_teachin~t |   .0208071   .0954981     0.22   0.828    -.1692041    .2108183
         ks2 |    .074796   .0249081     3.00   0.004     .0252366    .1243554
         fsm |  -.0084845   .0043366    -1.96   0.054    -.0171129    .0001439
         eal |    .011652   .0020678     5.64   0.000     .0075378    .0157663
         sen |   .0097616    .005411     1.80   0.075    -.0010046    .0205279
    log_size |  -.0317555   .1254432    -0.25   0.801    -.2813482    .2178371
years_sinc~d |  -.0108238   .0112597    -0.96   0.339     -.033227    .0115795
     academy |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1754281   .0818554     2.14   0.035     .0125614    .3382947
gs_strictn~t |   .1735591   .0589917     2.94   0.004     .0561842     .290934
gs_teachin~t |   -.021361   .0992688    -0.22   0.830    -.2188748    .1761528
         ks2 |   .1163769     .03578     3.25   0.002      .045186    .1875679
         fsm |  -.0055886   .0049354    -1.13   0.261    -.0154085    .0042314
         eal |   .0129387   .0022802     5.67   0.000     .0084018    .0174755
         sen |    .008221   .0065516     1.25   0.213    -.0048146    .0212566
    log_size |  -.0618285    .146998    -0.42   0.675    -.3543084    .2306515
years_sinc~d |  -.0095869   .0123993    -0.77   0.442    -.0342576    .0150837
     academy |


Linear regression                               Number of obs     =         96
                                                F(14, 81)         =      11.60
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5705
                                                Root MSE          =     .35837

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1018695   .0752271     1.35   0.179    -.0478089    .2515478
gs_strictn~t |   .1169107   .0658096     1.78   0.079    -.0140297     .247851
gs_teachin~t |     .08598   .1029555     0.84   0.406    -.1188692    .2908293
         ks2 |    .115907   .0278502     4.16   0.000     .0604939    .1713201
         fsm

  -.1349306   1.110933    -0.12   0.904    -2.345338    2.075477
             |
       _cons |  -14.23557   3.161353    -4.50   0.000    -20.52567   -7.945467
------------------------------------------------------------------------------

Stage 3 — Open (W+S+T, n=96):
 β_W =  0.102 (se= 0.075) p=0.179
 β_S =  0.117 (se= 0.066) p=0.079
 γ_T =  0.086 (se= 0.103) p=0.406
 R² = 0.5705


In [7]:
* ================================================================
* Attenuation summary: Stage 1 → Stage 3 coefficient retention
* ================================================================
display _newline "=== Attenuation: Stage 1 β retained in Stage 3 ==="
display " Outcome     β_W(S1)  β_W(S3)  %ret   β_S(S1)  β_S(S3)  %ret"
display " -----------------------------------------------------------------"

foreach lbl in Overall English Maths EBaC Open {
    estimates restore s1_`lbl'
    local bw1 = _b[gs_warmth_visit]
    local bs1 = _b[gs_strictness_visit]
    
    estimates restore s3_`lbl'
    local bw3 = _b[gs_warmth_visit]
    local bs3 = _b[gs_strictness_visit]
    
    local retW = cond(`bw1'!=0, `bw3'/`bw1'*100, .)
    local retS = cond(`bs1'!=0, `bs3'/`bs1'*100, .)
    
    display " `lbl': " ///
        %7.3f `bw1' "  " %7.3f `bw3' "  " %5.0f `retW' "%%" ///
        "   " %7.3f `bs1' "  " %7.3f `bs3' "  " %5.0f `retS' "%"
}


=== Attenuation: Stage 1 β retained in Stage 3 ===
 Outcome β_W(S1) β_W(S3) %ret β_S(S1) β_S(S3) %ret
 -----------------------------------------------------------------
(results s1_Overall are active now)
(results s3_Overall are active now)
 Overall:   0.150   0.141    94%%   0.130   0.126    97%
(results s1_English are active now)
(results s3_English are active now)
 English:   0.177   0.186   106%%   0.086   0.090   105%
(results s1_Maths are active now)
(results s3_Maths are active now)


 Maths:   0.118   0.107    91%%   0.109   0.105    96%
(results s1_EBaC are active now)
(results s3_EBaC are active now)
 EBaC:   0.165   0.175   107%%   0.169   0.174   103%
(results s1_Open are active now)
(results s3_Open are active now)
 Open:   0.146   0.102    70%%   0.135   0.117    87%


In [8]:
* ---- VIF check (last Stage 3 regression) ----
regress p8meaebac_avg $WS $T $controls if tier1, vce(hc3)
estat vif


Linear regression                               Number of obs     =         96
                                                F(14, 81)         =      10.59
                                                Prob > F          =     0.0000
                                                R-squared         =     0.6126
                                                Root MSE          =     .37275



------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1754281   .0818554     2.14   0.035     .0125614    .3382947
gs_strictn~t |   .1735591   .0589917     2.94   0.004     .0561842     .290934
gs_teachin~t |   -.021361   .0992688    -0.22   0.830    -.2188748    .1761528
         ks2 |   .1163769     .03578     3.25   0.002      .045186    .1875679
         fsm |  -.0055886   .0049354    -1.13   0.261    -.0154085    .0042314
         eal |   .0129387   .0022802     5.67   0.000     .0084018    .0174755
         sen |    .008221   .0065516     1.25   0.213    -.0048146    .0212566
    log_size |  -.0618285    .146998    -0.42   0.675    -.3543084    .2306515
years_sinc~d |  -.0095869   .0123993    -0.77   0.442    -.0342576    .0150837
     academy |


    Variable |       VIF       1/VIF  
-------------+----------------------


gs_warmth_~t |      3.36    0.297277
gs_strictn~t |      1.96    0.510723
gs_teachin~t |      4.08    0.245224
         ks2 |      3.95    0.252899
         fsm |      3.37    0.296814
         eal |      2.56    0.391381
         sen |      1.56    0.642233
    log_size |      1.30    0.768270
years_sinc~d |      1.26    0.795025
     academy |      1.18    0.846185
   urban_bin |      1.28    0.782983
   selective |      2.89    0.346556
ofsted_~2019 |
          3  |      1.41    0.711452
          4  |      1.10    0.908602
-------------+----------------------
    Mean VIF |      2.23


In [9]:
* ================================================================
* Export: tab_main_results.tex
* Three-panel table: Stage 1 (5 cols) | Stage 2 (5 cols) | Stage 3 (5 cols)
* ================================================================

* Panel A: Stage 1
esttab s1_Overall s1_English s1_Maths s1_EBaC s1_Open ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_main_results_s1.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_warmth_visit gs_strictness_visit) ///
    coeflabels(gs_warmth_visit "Warmth (\$W\$)" gs_strictness_visit "Strictness (\$S\$)") ///
    stats(N r2, labels("\$N\$" "\$R^2\$") fmt(%9.0f %8.3f)) ///
    mlabels("Overall" "English" "Maths" "EBaC" "Open") nonumbers ///
    title("Stage 1: Total culture effect (\$W + S\$)") ///
    b(%8.3f) se(%8.3f)

* Panel B: Stage 2
esttab s2_Overall s2_English s2_Maths s2_EBaC s2_Open ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_main_results_s2.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_teaching_visit) ///
    coeflabels(gs_teaching_visit "Teaching quality (\$T\$)") ///
    stats(N r2, labels("\$N\$" "\$R^2\$") fmt(%9.0f %8.3f)) ///
    mlabels("Overall" "English" "Maths" "EBaC" "Open") nonumbers ///
    title("Stage 2: Teaching quality benchmark (\$T\$)") ///
    b(%8.3f) se(%8.3f)

* Panel C: Stage 3
esttab s3_Overall s3_English s3_Maths s3_EBaC s3_Open ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_main_results_s3.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_warmth_visit gs_strictness_visit gs_teaching_visit) ///
    coeflabels(gs_warmth_visit "Warmth (\$W\$)" ///
               gs_strictness_visit "Strictness (\$S\$)" ///
               gs_teaching_visit "Teaching quality (\$T\$)") ///
    stats(N r2, labels("\$N\$" "\$R^2\$") fmt(%9.0f %8.3f)) ///
    mlabels("Overall" "English" "Maths" "EBaC" "Open") nonumbers ///
    title("Stage 3: Culture net of teaching (\$W + S + T\$)") ///
    b(%8.3f) se(%8.3f)

display "Main results tables written."

(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_main_results_s1.tex)


(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_main_results_s2.tex)


(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_main_results_s3.tex)
Main results tables written.


In [10]:
* ================================================================
* ROBUSTNESS 1: No pre-COVID Ofsted grade (all 102 schools)
* ================================================================
estimates clear

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    regress `outcome' $WS $controls_ngrade if tier1, vce(hc3)
    estimates store rob_ngrade_`lbl'
    display "No-grade Stage 1 — `lbl' (n=" e(N) "): β_W=" %6.3f _b[gs_warmth_visit] ///
        " β_S=" %6.3f _b[gs_strictness_visit]
}

display _newline "(Compare to primary spec with Ofsted grade controls above)"


Linear regression                               Number of obs     =        102
                                                F(11, 90)         =      14.25
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5759
                                                Root MSE          =     .32983



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1296451   .0472684     2.74   0.007     .0357382    .2235519
gs_strictn~t |   .1247574   .0468666     2.66   0.009     .0316487    .2178661
         ks2 |   .0967798   .0259348     3.73   0.000     .0452558    .1483038
         fsm |   -.005829   .0039493    -1.48   0.143     -.013675    .0020169
         eal |   .0108465   .0017045     6.36   0.000     .0074602    .0142329
         sen |   .0058146   .0052588     1.11   0.272    -.0046329    .0162621
    log_size |   .0946131   .1184774     0.80   0.427    -.1407629    .3299891
years_sinc~d |  -.0027409   .0096934    -0.28   0.778    -.0219984    .0165167
     academy |   .0456112   .0977911     0.47   0.642     -.148668    .2398904
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1100694   .0521653     2.11   0.038     .0064339    .2137048
gs_strictn~t |   .1012892   .0488983     2.07   0.041     .0041441    .1984343
         ks2 |   .0760651    .021964     3.46   0.001     .0324299    .1197004
         fsm |  -.0082001   .0037946    -2.16   0.033    -.0157387   -.0006614
         eal |   .0109082   .0016748     6.51   0.000     .0075809    .0142355
         sen |   .0091215   .0055273     1.65   0.102    -.0018595    .0201024
    log_size |   .0375283   .1130503     0.33   0.741    -.1870659    .2621225
years_sinc~d |  -.0038293    .010205    -0.38   0.708    -.0241034    .0164448
     academy |  -.0144225   .0999809    -0.14   0.886    -.2130521    .1842071
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1502177   .0568475     2.64   0.010     .0372803    .2631552
gs_strictn~t |    .155618   .0581183     2.68   0.009     .0401559    .2710801
         ks2 |   .1031669   .0317548     3.25   0.002     .0400804    .1662534
         fsm |    -.00625   .0046352    -1.35   0.181    -.0154586    .0029587
         eal |   .0127355    .002164     5.89   0.000     .0084362    .0170347
         sen |   .0071885   .0066174     1.09   0.280    -.0059581     .020335
    log_size |   .0662529   .1486592     0.45   0.657    -.2290844    .3615903
years_sinc~d |  -.0006865   .0112226    -0.06   0.951    -.0229822    .0216092
     academy |   .0651794   .1175193     0.55   0.581    -.1682932    .2986519
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1166145    .062992     1.85   0.067      -.00853     .241759
gs_strictn~t |   .1368107   .0648265     2.11   0.038     .0080215       .2656
         ks2 |   .0976653   .0304713     3.21   0.002     .0371287    .1582018
         fsm |  -.0049806   .0047312    -1.05   0.295      -.01438    .0044188
         eal |   .0074171   .0024601     3.01   0.003     .0025298    .0123045
         sen |   .0016434   .0066145     0.25   0.804    -.0114974    .0147842
    log_size |   .2218721   .1386204     1.60   0.113    -.0535215    .4972658
years_sinc~d |  -.0035877   .0112051    -0.32   0.750    -.0258485    .0186731
     academy |   .0328432   .1187681     0.28   0.783    -.2031103    .2687967
   urban_bin |

In [11]:
* ================================================================
* ROBUSTNESS 2: Single year 2023-24 P8 (overall + components)
* ================================================================

foreach outcome in p8mea_2324 p8meaeng_2324 p8meamat_2324 p8meaebac_2324 p8meaopen_2324 {
    local lbl = cond("`outcome'"=="p8mea_2324",    "Overall", ///
                cond("`outcome'"=="p8meaeng_2324",  "English", ///
                cond("`outcome'"=="p8meamat_2324",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_2324", "EBaC",    "Open"))))
    
    regress `outcome' $WS $controls if tier1, vce(hc3)
    estimates store rob_2324_`lbl'
    display "2023-24 single year — `lbl' (n=" e(N) "): β_W=" %6.3f _b[gs_warmth_visit] ///
        " β_S=" %6.3f _b[gs_strictness_visit]
}


Linear regression                               Number of obs     =         96
                                                F(12, 82)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.6515
                                                Root MSE          =     .31017



------------------------------------------------------------------------------
             |             Robust HC3
  p8mea_2324 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1654413    .046141     3.59   0.001     .0736522    .2572304
gs_strictn~t |   .1246138   .0451178     2.76   0.007     .0348601    .2143675
         ks2 |   .1084654   .0276951     3.92   0.000      .053371    .1635599
         fsm |  -.0070068   .0041334    -1.70   0.094    -.0152295    .0012159
         eal |   .0113678   .0017857     6.37   0.000     .0078155      .01492
         sen |    .005133   .0050466     1.02   0.312    -.0049062    .0151722
    log_size |  -.0863647   .0964471    -0.90   0.373    -.2782287    .1054992
years_sinc~d |  -.0080831   .0105609    -0.77   0.446    -.0290922     .012926
     academy |   .0200538   .0923333     0.22   0.829    -.1636266    .2037343
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaen~2324 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |    .181157   .0462898     3.91   0.000     .0890719    .2732421
gs_strictn~t |   .0793743   .0521855     1.52   0.132    -.0244394    .1831879
         ks2 |   .1188993   .0307175     3.87   0.000     .0577924    .1800062
         fsm |  -.0045341    .004732    -0.96   0.341    -.0139476    .0048795
         eal |   .0135746   .0021831     6.22   0.000     .0092317    .0179175
         sen |   .0027607   .0054235     0.51   0.612    -.0080283    .0135497
    log_size |  -.1561502   .1158788    -1.35   0.182      -.38667    .0743696
years_sinc~d |  -.0063287   .0116855    -0.54   0.590     -.029575    .0169175
     academy |   .0400783   .0998369     0.40   0.689     -.158529    .2386856
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meama~2324 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |    .136846   .0592844     2.31   0.024     .0189105    .2547816
gs_strictn~t |   .0876545   .0523274     1.68   0.098    -.0164413    .1917503
         ks2 |   .0696615   .0267999     2.60   0.011     .0163479     .122975
         fsm |  -.0099808     .00443    -2.25   0.027    -.0187934   -.0011682
         eal |   .0113809   .0020166     5.64   0.000     .0073692    .0153926
         sen |    .009153   .0059216     1.55   0.126    -.0026269    .0209328
    log_size |  -.0674669   .1127554    -0.60   0.551    -.2917733    .1568396
years_sinc~d |  -.0091469   .0116212    -0.79   0.433    -.0322652    .0139714
     academy |  -.0335878   .1043581    -0.32   0.748    -.2411893    .1740136
   urban_bin |


Linear regression                               Number of obs     =         96
                                                F(13, 82)         =      11.40
                                                Prob > F          =     0.0000
                                                R-squared         =     0.6125
                                                Root MSE          =     .38409

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeb~2324 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1707733   .0600709     2.84   0.006     .0512732    .2902733
gs_strictn~t |   .1809993   .0583695     3.10   0.003     .0648838    .2971148
         ks2 |   .1220516   .0364912     3.34   0.001     .0494589    .1946443
         fsm |   -.006452   .0048309    -1.34   0.185    -.0160622    .0031583
         eal

  -.2075389   .1677776    -1.24   0.220    -.5413021    .1262242
          4  |  -.1803697   .3154847    -0.57   0.569    -.8079692    .4472298
             |
       _cons |  -14.38864   3.920011    -3.67   0.000    -22.18679   -6.590487
------------------------------------------------------------------------------
2023-24 single year — EBaC (n=96): β_W= 0.171 β_S= 0.181

Linear regression                               Number of obs     =         96
                                                F(12, 82)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5841
                                                Root MSE          =     .36683



------------------------------------------------------------------------------
             |             Robust HC3
p8meaop~2324 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1727585   .0603406     2.86   0.005     .0527217    .2927952
gs_strictn~t |   .1232074   .0645671     1.91   0.060    -.0052371    .2516519
         ks2 |   .1140851   .0292658     3.90   0.000     .0558662    .1723041
         fsm |  -.0076645   .0050705    -1.51   0.134    -.0177513    .0024224
         eal |   .0089432   .0027156     3.29   0.001      .003541    .0143454
         sen |   .0029084   .0066105     0.44   0.661     -.010242    .0160588
    log_size |  -.0464271   .1103454    -0.42   0.675    -.2659393     .173085
years_sinc~d |  -.0106244   .0120147    -0.88   0.379    -.0345254    .0132765
     academy |   .0182108   .1171518     0.16   0.877    -.2148415     .251263
   urban_bin |

In [12]:
* ================================================================
* ROBUSTNESS 3: Att8 2024-25 components + total (contemporaneous)
* att8_total = sum of 4 bucket scores (already destringed in cell-load)
* ================================================================

gen att8_total_2425 = att8screng_2425 + att8scrmat_2425 + att8screbac_2425 + att8scropen_2425

foreach outcome in att8_total_2425 att8screng_2425 att8scrmat_2425 att8screbac_2425 att8scropen_2425 {
    local lbl = cond("`outcome'"=="att8_total_2425",  "Overall", ///
                cond("`outcome'"=="att8screng_2425",  "English", ///
                cond("`outcome'"=="att8scrmat_2425",  "Maths",   ///
                cond("`outcome'"=="att8screbac_2425", "EBaC",    "Open"))))
    
    regress `outcome' $WS $controls if tier1, vce(hc3)
    estimates store rob_att8_`lbl'
    display "Att8 2024-25 — `lbl' (n=" e(N) "): β_W=" %6.3f _b[gs_warmth_visit] ///
        " β_S=" %6.3f _b[gs_strictness_visit]
}

display "(Note: Att8 does not subtract KS2 prior attainment — higher R² expected)"

(58 missing values generated)

Linear regression                               Number of obs     =         96
                                                F(13, 82)         =      65.34
                                                Prob > F          =     0.0000
                                                R-squared         =     0.8762
                                                Root MSE          =     3.7479



------------------------------------------------------------------------------
             |             Robust HC3
att8_to~2425 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   1.272676    .517021     2.46   0.016     .2441564    2.301196
gs_strictn~t |   1.691117   .5009815     3.38   0.001     .6945057    2.687729
         ks2 |   3.080254   .2743277    11.23   0.000     2.534529     3.62598
         fsm |  -.0983296   .0522645    -1.88   0.063    -.2023002    .0056411
         eal |   .1143281   .0242054     4.72   0.000     .0661759    .1624803
         sen |   .1297772   .0592261     2.19   0.031     .0119577    .2475967
    log_size |  -.3476943   1.570386    -0.22   0.825    -3.471692    2.776303
years_sinc~d |  -.1289189   .1148101    -1.12   0.265    -.3573128    .0994751
     academy |  -1.240886   1.097323    -1.13   0.261    -3.423812    .9420392
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
att8screng~5 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |     .33711   .1012814     3.33   0.001      .135629     .538591
gs_strictn~t |    .218004   .1062901     2.05   0.043     .0065591    .4294489
         ks2 |   .5778112   .0589901     9.80   0.000     .4604612    .6951613
         fsm |  -.0160809   .0115554    -1.39   0.168    -.0390682    .0069064
         eal |   .0261891   .0049195     5.32   0.000     .0164026    .0359756
         sen |   .0246761   .0136148     1.81   0.074    -.0024082    .0517603
    log_size |  -.3967151   .3056376    -1.30   0.198    -1.004726    .2112954
years_sinc~d |  -.0174882   .0261369    -0.67   0.505    -.0694829    .0345065
     academy |  -.0979541   .2372891    -0.41   0.681    -.5699977    .3740896
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
att8scrmat~5 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1086573    .137764     0.79   0.433    -.1653992    .3827138
gs_strictn~t |   .4121509   .1289926     3.20   0.002     .1555435    .6687583
         ks2 |   .6082918   .0571901    10.64   0.000     .4945224    .7220611
         fsm |  -.0188066   .0106044    -1.77   0.080    -.0399021    .0022889
         eal |    .018272   .0058297     3.13   0.002      .006675    .0298691
         sen |    .027865   .0145452     1.92   0.059    -.0010701       .0568
    log_size |    .115154   .3465949     0.33   0.741    -.5743337    .8046417
years_sinc~d |  -.0156902   .0260416    -0.60   0.549    -.0674953    .0361149
     academy |  -.2367824   .2564172    -0.92   0.358     -.746878    .2733132
   urban_bin |


Linear regression                               Number of obs     =         96
                                                F(13, 82)         =      50.07
                                                Prob > F          =     0.0000
                                                R-squared         =     0.8579
                                                Root MSE          =     1.3319

------------------------------------------------------------------------------
             |             Robust HC3
att8screba~5 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .3462966   .1768717     1.96   0.054    -.0055576    .6981508
gs_strictn~t |   .6706025   .1860833     3.60   0.001     .3004236    1.040781
         ks2 |   1.009208   .1065493     9.47   0.000     .7972475    1.221168
         fsm |  -.0316263   .0180244    -1.75   0.083    -.0674826    .0042299
         eal

years_sinc~d |  -.0377085   .0392939    -0.96   0.340    -.1158767    .0404596
     academy |  -.4215959   .3812662    -1.11   0.272    -1.180056    .3368641
   urban_bin |  -.3075286   .4667531    -0.66   0.512    -1.236049     .620992
   selective |  -1.691155   1.548077    -1.09   0.278    -4.770774    1.388464
             |
ofsted_~2019 |
          3  |   -.178006   .3708953    -0.48   0.633    -.9158348    .5598228
          4  |  -.7372114   2.335696    -0.32   0.753    -5.383656    3.909233
             |
       _cons |  -96.94549   12.15827    -7.97   0.000    -121.1322   -72.75882
------------------------------------------------------------------------------
Att8 2024-25 — EBaC (n=96): β_W= 0.346 β_S= 0.671

Linear regression                               Number of obs     =         96
                                                F(13, 82)         =      52.03
                                                Prob > F          =     0.0000
                                   

------------------------------------------------------------------------------
             |             Robust HC3
att8scrope~5 | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |    .480612   .1723243     2.79   0.007      .137804      .82342
gs_strictn~t |   .3903601   .1611421     2.42   0.018     .0697972     .710923
         ks2 |   .8849433   .0820443    10.79   0.000     .7217309    1.048156
         fsm |  -.0318157   .0170051    -1.87   0.065    -.0656442    .0020129
         eal |   .0281083   .0088014     3.19   0.002     .0105995     .045617
         sen |   .0352953   .0178928     1.97   0.052    -.0002992    .0708898
    log_size |     .08268   .4565642     0.18   0.857    -.8255716    .9909316
years_sinc~d |  -.0580319   .0333792    -1.74   0.086    -.1244337      .00837
     academy |  -.4845539   .3393143    -1.43   0.157    -1.159558    .1904504
   urban_bin |

In [13]:
* ================================================================
* ROBUSTNESS 4: Continuity-restricted sample
* (ofsted_HeadteacherChanged == 0: HT unchanged since Ofsted inspection)
* Note: only 30 of 102 visited schools have this variable; not missing at random
* ================================================================

* ofsted_headteacherchanged is stored as "True"/"False" strings in the CSV
gen ht_changed = .
replace ht_changed = 1 if lower(ofsted_headteacherchanged) == "true"
replace ht_changed = 0 if lower(ofsted_headteacherchanged) == "false"

count if tier1 & ht_changed == 0
count if tier1 & ht_changed == 1
count if tier1 & missing(ht_changed)

foreach outcome in p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8meaeng_avg", "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open")))
    
    * Note: using controls_ngrade to avoid further sample reduction from missing grade
    regress `outcome' $WS $controls_ngrade if tier1 & ht_changed==0, vce(hc3)
    estimates store rob_cont_`lbl'
    display "Continuity (unchanged HT) — `lbl' (n=" e(N) "): β_W=" %6.3f _b[gs_warmth_visit] ///
        " β_S=" %6.3f _b[gs_strictness_visit]
}

display _newline "Caution: n is very small in this restricted sample — interpret as directional only."

(3,332 missing values generated)
(636 real changes made)
(780 real changes made)
  22
  7
  74
note: selective omitted because of collinearity.

Linear regression                               Number of obs     =         22
                                                F(10, 11)         =       4.77
                                                Prob > F          =     0.0083
                                                R-squared         =     0.7577
                                                Root MSE          =     .32702



------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .0415879   .2358123     0.18   0.863    -.4774314    .5606072
gs_strictn~t |   .2322103   .1869002     1.24   0.240    -.1791543    .6435749
         ks2 |   .1197665   .0862955     1.39   0.193    -.0701685    .3097015
         fsm |   -.005108   .0137281    -0.37   0.717    -.0353233    .0251074
         eal |   .0210124   .0060708     3.46   0.005     .0076506    .0343741
         sen |   .0323701    .031317     1.03   0.323     -.036558    .1012983
    log_size |  -.3797139    .513358    -0.74   0.475    -1.509607    .7501793
years_sinc~d |  -.0158375    .029006    -0.55   0.596    -.0796793    .0480044
     academy |   .2320561   .3845041     0.60   0.558    -.6142318    1.078344
   urban_bin |

          0  (omitted)
       _cons |  -12.54198    9.99303    -1.26   0.235    -34.53649    9.452535
------------------------------------------------------------------------------
Continuity (unchanged HT) — English (n=22): β_W= 0.042 β_S= 0.232
note: selective omitted because of collinearity.

Linear regression                               Number of obs     =         22
                                                F(10, 11)         =       0.65
                                                Prob > F          =     0.7500
                                                R-squared         =     0.5300
                                                Root MSE          =      .3319



------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .0101601   .2255707     0.05   0.965    -.4863177    .5066378
gs_strictn~t |    .139899   .1927192     0.73   0.483     -.284273     .564071
         ks2 |  -.0270428   .1026567    -0.26   0.797    -.2529887    .1989032
         fsm |  -.0125133   .0148916    -0.84   0.419    -.0452894    .0202629
         eal |   .0075964   .0065998     1.15   0.274    -.0069298    .0221225
         sen |   .0083019   .0306624     0.27   0.792    -.0591855    .0757894
    log_size |  -.3533431   .5487059    -0.64   0.533    -1.561037    .8543505
years_sinc~d |   .0083656   .0312841     0.27   0.794    -.0604902    .0772213
     academy |   .3323812   .4066291     0.82   0.431    -.5626035    1.227366
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   -.014625   .2635904    -0.06   0.957    -.5947836    .5655335
gs_strictn~t |   .1899877   .2817244     0.67   0.514    -.4300835    .8100588
         ks2 |   .0890681   .1295362     0.69   0.506    -.1960392    .3741755
         fsm |  -.0050971   .0179551    -0.28   0.782    -.0446159    .0344217
         eal |   .0119336   .0099168     1.20   0.254    -.0098931    .0337603
         sen |   .0222583   .0309233     0.72   0.487    -.0458034      .09032
    log_size |  -.3568984   .8284878    -0.43   0.675    -2.180388    1.466591
years_sinc~d |  -.0031828   .0413022    -0.08   0.940    -.0940882    .0877227
     academy |   .2754415   .5564127     0.50   0.630    -.9492147    1.500098
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |  -.0267599   .2264226    -0.12   0.908    -.5251126    .4715929
gs_strictn~t |   .2893663   .2723656     1.06   0.311    -.3101062    .8888389
         ks2 |   .0828353   .1308459     0.63   0.540    -.2051546    .3708251
         fsm |  -.0071823   .0183915    -0.39   0.704    -.0476618    .0332972
         eal |   .0172416    .006495     2.65   0.022     .0029462     .031537
         sen |   .0002125   .0378032     0.01   0.996    -.0829916    .0834167
    log_size |  -.6825382   .7946217    -0.86   0.409    -2.431489    1.066412
years_sinc~d |   .0016363   .0446156     0.04   0.971     -.096562    .0998345
     academy |   .1112775   .5963031     0.19   0.855    -1.201177    1.423732
   urban_bin |

In [14]:
* ================================================================
* ROBUSTNESS 5: W × S interaction term
* ================================================================

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    regress `outcome' c.gs_warmth_visit##c.gs_strictness_visit $controls if tier1, vce(hc3)
    estimates store rob_inter_`lbl'
    display "W×S interaction — `lbl' (n=" e(N) "): β_W=" %6.3f _b[gs_warmth_visit] ///
        " β_S=" %6.3f _b[gs_strictness_visit] ///
        " β_WxS=" %6.3f _b[c.gs_warmth_visit#c.gs_strictness_visit] ///
        " p_inter=" %5.3f (2*ttail(e(df_r), abs(_b[c.gs_warmth_visit#c.gs_strictness_visit]/_se[c.gs_warmth_visit#c.gs_strictness_visit])))
}


Linear regression                               Number of obs     =         96
                                                F(14, 81)         =      14.58
                                                Prob > F          =     0.0000
                                                R-squared         =     0.6560
                                                Root MSE          =      .3009



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |  -.1231785    .253885    -0.49   0.629    -.6283299    .3819729
gs_strictn~t |  -.1296443   .2315366    -0.56   0.577    -.5903295    .3310408
             |
          c. |
gs_warmth_~t#|
          c. |
gs_strictn~t |   .0397207   .0365903     1.09   0.281    -.0330825     .112524
             |
         ks2 |   .1084642    .026928     4.03   0.000     .0548859    .1620425
         fsm |  -.0055034   .0038113    -1.44   0.153    -.0130866    .0020798
         eal |   .0109365   .0018559     5.89   0.000     .0072437    .0146292
         sen |    .006435   .0051923     1.24   0.219     -.003896    .0167659
    log_size |  -.0241572    .103474    -0.23   0.816    -.2300381    .1817237
years_sinc~d |  -.


Linear regression                               Number of obs     =         96
                                                F(13, 81)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.6400
                                                Root MSE          =     .32846

------------------------------------------------------------------------------
             |             Robust HC3


p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .0157791   .2853973     0.06   0.956     -.552072    .5836302
gs_strictn~t |  -.0661303   .2619225    -0.25   0.801     -.587274    .4550134
             |
          c. |
gs_warmth_~t#|
          c. |
gs_strictn~t |   .0233557   .0403166     0.58   0.564    -.0568616    .1035731
             |
         ks2 |   .1222482   .0303399     4.03   0.000     .0618814    .1826151
         fsm |  -.0042971    .004488    -0.96   0.341    -.0132269    .0046326
         eal |   .0138592    .002163     6.41   0.000     .0095554    .0181629
         sen |   .0062195   .0054073     1.15   0.253    -.0045394    .0169784
    log_size |  -.1207989   .1068749    -1.13   0.262    -.3334464    .0918486
years_sinc~d |  -.0081546   .0111554    -0.73   0.467    -.0303504    .0140413
     academy |   .0889817   .1034984     0.86   0.392   

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |  -.1644276   .2968827    -0.55   0.581    -.7551312    .4262759
gs_strictn~t |  -.1582179   .2891601    -0.55   0.586    -.7335559      .41712
             |
          c. |
gs_warmth_~t#|
          c. |
gs_strictn~t |   .0409697   .0436025     0.94   0.350    -.0457856    .1277251
             |
         ks2 |   .0743451   .0245837     3.02   0.003     .0254313    .1232588
         fsm |  -.0080189   .0038614    -2.08   0.041    -.0157019   -.0003359
         eal |   .0111007   .0019017     5.84   0.000     .0073169    .0148845
         sen |   .0091407   .0055353     1.65   0.103    -.0018728    .0201542
    log_size |  -.0114682   .1177643    -0.10   0.923    -.2457822    .2228457
years_sinc~d |  -.


Linear regression                               Number of obs     =         96
                                                F(14, 81)         =      11.84
                                                Prob > F          =     0.0000
                                                R-squared         =     0.6224
                                                Root MSE          =       .368

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   -.251987   .2897778    -0.87   0.387     -.828554    .3245799
gs_strictn~t |  -.2261333   .2507611    -0.90   0.370    -.7250693    .2728027
             |
          c. |
gs_warmth_~t#|
          c. |
gs_strictn~t |   .0604988   .0410508     1.47   0.144    -.0211793     .142177
             |
         ks2 |  


Linear regression                               Number of obs     =         96
                                                F(14, 81)         =      11.47
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5677
                                                Root MSE          =     .35952



------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |  -.0404736   .3197928    -0.13   0.900     -.676761    .5958137
gs_strictn~t |  -.0416179   .2982671    -0.14   0.889    -.6350759    .5518401
             |
          c. |
gs_warmth_~t#|
          c. |
gs_strictn~t |   .0270421   .0463626     0.58   0.561    -.0652049    .1192891
             |
         ks2 |    .119678   .0283068     4.23   0.000     .0633563    .1759997
         fsm |  -.0041133    .004866    -0.85   0.400    -.0137952    .0055686
         eal |   .0075326   .0029295     2.57   0.012     .0017038    .0133614
         sen |   .0031885   .0067231     0.47   0.637    -.0101883    .0165654
    log_size |   .0510931    .119246     0.43   0.669    -.1861691    .2883554
years_sinc~d |  -.

In [15]:
* ================================================================
* ROBUSTNESS 6: Add SEMH baseline 2016 as additional control
* ================================================================
destring semh_baseline_2016, replace force

count if tier1 & !missing(semh_baseline_2016)

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    regress `outcome' $WS semh_baseline_2016 $controls if tier1, vce(hc3)
    estimates store rob_semh_`lbl'
    display "SEMH control — `lbl' (n=" e(N) "): β_W=" %6.3f _b[gs_warmth_visit] ///
        " β_S=" %6.3f _b[gs_strictness_visit]
}

semh_baseline_2016 already numeric; no replace
  96

Linear regression                               Number of obs     =         91
                                                F(14, 76)         =      13.42
                                                Prob > F          =     0.0000
                                                R-squared         =     0.6592
                                                Root MSE          =     .29005



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1631572   .0438539     3.72   0.000     .0758146    .2504997
gs_strictn~t |   .1053172   .0401153     2.63   0.010     .0254207    .1852137
semh_ba~2016 |   -.003089   .0018318    -1.69   0.096    -.0067374    .0005593
         ks2 |    .101641   .0235508     4.32   0.000     .0547354    .1485465
         fsm |   -.007791     .00353    -2.21   0.030    -.0148216   -.0007604
         eal |   .0135577   .0017002     7.97   0.000     .0101715     .016944
         sen |    .010918     .00527     2.07   0.042      .000422    .0214141
    log_size |  -.0575484   .1005919    -0.57   0.569    -.2578945    .1427977
years_sinc~d |   -.017769   .0102717    -1.73   0.088    -.0382267    .0026888
     academy |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1784738   .0418225     4.27   0.000      .095177    .2617705
gs_strictn~t |    .074013   .0447484     1.65   0.102    -.0151111    .1631372
semh_ba~2016 |  -.0047892   .0015077    -3.18   0.002    -.0077922   -.0017863
         ks2 |   .1212534   .0258717     4.69   0.000     .0697255    .1727813
         fsm |  -.0064664   .0042278    -1.53   0.130    -.0148868     .001954
         eal |   .0173881   .0020273     8.58   0.000     .0133504    .0214258
         sen |   .0116952   .0055158     2.12   0.037     .0007096    .0226808
    log_size |  -.1438688   .1013782    -1.42   0.160    -.3457809    .0580433
years_sinc~d |  -.0166478   .0115467    -1.44   0.153     -.039645    .0063494
     academy |


Linear regression                               Number of obs     =         91
                                                F(14, 76)         =       6.34
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5146
                                                Root MSE          =     .33376

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |    .132322   .0588078     2.25   0.027     .0151962    .2494479
gs_strictn~t |   .0766089   .0528497     1.45   0.151    -.0286505    .1818683
semh_ba~2016 |  -.0027689   .0015292    -1.81   0.074    -.0058145    .0002767
         ks2 |   .0632291     .02529     2.50   0.015     .0128597    .1135986
         fsm

SEMH control — Maths (n=91): β_W= 0.132 β_S= 0.077

Linear regression                               Number of obs     =         91
                                                F(13, 76)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.6168
                                                Root MSE          =     .36119

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1701228    .060013     2.83   0.006     .0505966    .2896491
gs_strictn~t |

   .1492443   .0512602     2.91   0.005     .0471508    .2513378
semh_ba~2016 |  -.0024379   .0026101    -0.93   0.353    -.0076364    .0027605
         ks2 |   .1129832   .0317502     3.56   0.001     .0497472    .1762192
         fsm |   -.008086   .0044535    -1.82   0.073     -.016956    .0007839
         eal |   .0149112    .002195     6.79   0.000     .0105395    .0192828
         sen |   .0109724   .0067498     1.63   0.108    -.0024709    .0244158
    log_size |  -.1097791   .1381919    -0.79   0.429    -.3850121    .1654539
years_sinc~d |  -.0154451    .012514    -1.23   0.221    -.0403689    .0094787
     academy |   .0494299   .1067399     0.46   0.645    -.1631611     .262021
   urban_bin |   .0071644   .1577335     0.05   0.964    -.3069891     .321318
   selective |  -.5711867   .9526353    -0.60   0.551    -2.468524    1.326151
             |
ofsted_~2019 |
          3  |  -.2289984   .1484125    -1.54   0.127    -.5245875    .0665906
          4  |  -.0611081    .267452

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |    .171682   .0567001     3.03   0.003      .058754    .2846099
gs_strictn~t |   .1030058   .0594824     1.73   0.087    -.0154637    .2214753
semh_ba~2016 |  -.0029592   .0024189    -1.22   0.225    -.0077768    .0018584
         ks2 |   .1053491   .0261197     4.03   0.000     .0533272     .157371
         fsm |   -.007303   .0045079    -1.62   0.109    -.0162813    .0016753
         eal |   .0105002   .0031639     3.32   0.001     .0041987    .0168018
         sen |    .008404   .0066923     1.26   0.213    -.0049248    .0217329
    log_size |   .0214841   .1155133     0.19   0.853    -.2085806    .2515487
years_sinc~d |  -.0214137   .0107266    -2.00   0.049    -.0427775   -.0000498
     academy |

In [16]:
* ================================================================
* Export: tab_robustness_overall.tex + tab_robustness_eng.tex
* Warmth and Strictness coefficients across robustness specs
* ================================================================

* Re-run primary (with Ofsted grade controls) alongside robustness specs
foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    regress `outcome' $WS $controls if tier1, vce(hc3)
    estimates store primary_`lbl'
}

* Overall P8 robustness table
esttab primary_Overall rob_ngrade_Overall rob_2324_Overall rob_att8_Overall ///
       rob_inter_Overall rob_semh_Overall ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_robustness_overall.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_warmth_visit gs_strictness_visit) ///
    coeflabels(gs_warmth_visit "\$W\$" gs_strictness_visit "\$S\$") ///
    mtitles("Primary" "No grade" "2023-24" "Att8 2425" "W\$\times\$S" "SEMH ctrl") ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) ///
    title("Robustness: Overall Progress 8")

* English P8 robustness table
esttab primary_English rob_ngrade_English rob_2324_English rob_att8_English ///
       rob_inter_English rob_semh_English ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_robustness_eng.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_warmth_visit gs_strictness_visit) ///
    coeflabels(gs_warmth_visit "\$W\$" gs_strictness_visit "\$S\$") ///
    mtitles("Primary" "No grade" "2023-24" "Att8 2425" "W\$\times\$S" "SEMH ctrl") ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) ///
    title("Robustness: English P8 component")

display "Robustness tables (Overall + English) written."


Linear regression                               Number of obs     =         96
                                                F(13, 82)         =      15.48
                                                Prob > F          =     0.0000
                                                R-squared         =     0.6501
                                                Root MSE          =     .30162



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1502921   .0447124     3.36   0.001     .0613448    .2392393
gs_strictn~t |   .1298203   .0468298     2.77   0.007     .0366608    .2229797
         ks2 |   .1100375   .0269014     4.09   0.000      .056522    .1635529
         fsm |  -.0054845   .0038657    -1.42   0.160    -.0131745    .0022056
         eal |   .0112759   .0017424     6.47   0.000     .0078097    .0147421
         sen |   .0070134   .0050614     1.39   0.170    -.0030554    .0170822
    log_size |  -.0385436   .1028519    -0.37   0.709    -.2431488    .1660615
years_sinc~d |  -.0103161    .010224    -1.01   0.316    -.0306549    .0100227
     academy |    .037192    .094407     0.39   0.695    -.1506135    .2249976
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1765795    .043382     4.07   0.000     .0902788    .2628801
gs_strictn~t |   .0864345   .0493691     1.75   0.084    -.0117763    .1846454
         ks2 |   .1231733   .0299718     4.11   0.000     .0635499    .1827968
         fsm |   -.004286   .0044701    -0.96   0.340    -.0131784    .0046064
         eal |   .0140587    .002112     6.66   0.000     .0098573    .0182602
         sen |   .0065597   .0052383     1.25   0.214    -.0038609    .0169802
    log_size |  -.1292581   .1065153    -1.21   0.228    -.3411511    .0826348
years_sinc~d |  -.0084408   .0110925    -0.76   0.449    -.0305073    .0136257
     academy |   .0816477   .0992325     0.82   0.413    -.1157574    .2790528
   urban_bin |


Linear regression                               Number of obs     =         96
                                                F(12, 82)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5378
                                                Root MSE          =     .33346

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1176421   .0556669     2.11   0.038      .006903    .2283812
gs_strictn~t |   .1094054   .0520123     2.10   0.038     .0059364    .2128744
         ks2 |   .0759678   .0243386     3.12   0.002     .0275507    .1243849
         fsm |  -.0079994    .003861    -2.07   0.041    -.0156801   -.0003186
         eal


Linear regression                               Number of obs     =         96
                                                F(12, 82)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.6123
                                                Root MSE          =      .3706



------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1645371   .0579617     2.84   0.006     .0492327    .2798414
gs_strictn~t |   .1690583   .0580863     2.91   0.005     .0535062    .2846104
         ks2 |    .115174   .0350296     3.29   0.001     .0454888    .1848591
         fsm |  -.0060866   .0045441    -1.34   0.184    -.0151262     .002953
         eal |   .0131453   .0021983     5.98   0.000     .0087723    .0175184
         sen |    .008246   .0064833     1.27   0.207    -.0046513    .0211433
    log_size |   -.067422   .1436475    -0.47   0.640    -.3531826    .2183386
years_sinc~d |  -.0093143   .0122223    -0.76   0.448    -.0336283    .0149998
     academy |   .0539755   .1130613     0.48   0.634    -.1709394    .2788903
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1457066   .0586771     2.48   0.015     .0289791    .2624341
gs_strictn~t |    .135027   .0646678     2.09   0.040     .0063821    .2636718
         ks2 |   .1207491   .0284083     4.25   0.000      .064236    .1772622
         fsm |  -.0041004   .0048353    -0.85   0.399    -.0137194    .0055186
         eal |   .0077636   .0027025     2.87   0.005     .0023874    .0131398
         sen |   .0035823   .0065785     0.54   0.588    -.0095044    .0166691
    log_size |   .0412988   .1112585     0.37   0.711    -.1800298    .2626273
years_sinc~d |  -.0124087   .0114994    -1.08   0.284    -.0352847    .0104674
     academy |   .0128434    .113271     0.11   0.910    -.2124888    .2381755
   urban_bin |

(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_robustness_overall.tex)


(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_robustness_eng.tex)
Robustness tables (Overall + English) written.


In [17]:
* ================================================================
* ENACTED vs ESPOUSED: Interview-only scores on 303 schools
* Uses gs_w3_adj (warmth interview) and mean(gs_s3, gs_s4) (strictness interview)
* Expected: lower coefficients than visit-only (espoused overclaims warmth)
* ================================================================
estimates clear

* Create interview-only scores (0-10 scale)
gen warmth_interview_only  = gs_w3_adj * 2
gen strict_interview_only  = (gs_s3 + gs_s4) / 2 * 2  if !missing(gs_s3) & !missing(gs_s4)

destring warmth_interview_only strict_interview_only, replace force

count if tier2 & !missing(warmth_interview_only) & !missing(strict_interview_only)

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    * Interview-only scores, n=267-303 (Tier 2, with pre-COVID grade where available)
    regress `outcome' warmth_interview_only strict_interview_only $controls if tier2, vce(hc3)
    estimates store espoused_`lbl'
    display _newline "Espoused (interview-only, n=" e(N) ") — `lbl':"
    display "  β_W = " %6.3f _b[warmth_interview_only] ///
            "  (se=" %6.3f _se[warmth_interview_only] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[warmth_interview_only]/_se[warmth_interview_only])))
    display "  β_S = " %6.3f _b[strict_interview_only] ///
            "  (se=" %6.3f _se[strict_interview_only] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[strict_interview_only]/_se[strict_interview_only])))
    display "  R² = " %6.4f e(r2)
}

(3,028 missing values generated)


(3,028 missing values generated)
warmth_interview_only already numeric; no replace
strict_interview_only already numeric; no replace
  304

Linear regression                               Number of obs     =        267
                                                F(13, 253)        =      20.54
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4332
                                                Root MSE          =     .36432

------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
warmth_int~y |   .0176614   .0141034     1.25   0.212    -.0101136    .0454364
strict_int~y |   .0622149   .0282252     2.20   0.028     .0066287    .1178012
         ks2 |    .067548   .01


Espoused (interview-only, n=267) — Overall:
 β_W =  0.018 (se= 0.014) p=0.212
 β_S =  0.062 (se= 0.028) p=0.028
 R² = 0.4332

Linear regression                               Number of obs     =        267
                                                F(13, 253)        =      13.57
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4133
                                                Root MSE          =     .37144



------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
warmth_int~y |   .0133252   .0153623     0.87   0.387    -.0169291    .0435795
strict_int~y |    .074798   .0281105     2.66   0.008     .0194376    .1301584
         ks2 |   .0673947    .017418     3.87   0.000     .0330921    .1016974
         fsm |  -.0090383   .0029292    -3.09   0.002    -.0148069   -.0032696
         eal |   .0116044    .001435     8.09   0.000     .0087784    .0144304
         sen |  -.0034382   .0037825    -0.91   0.364    -.0108875    .0040111
    log_size |   -.015252   .0652945    -0.23   0.815     -.143842     .113338
years_sinc~d |   .0064743   .0062063     1.04   0.298    -.0057483     .018697
     academy |  -.0200822   .0568628    -0.35   0.724    -.1320669    .0919025
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
warmth_int~y |   .0147885   .0146124     1.01   0.312    -.0139888    .0435659
strict_int~y |   .0281308   .0285781     0.98   0.326    -.0281505    .0844121
         ks2 |   .0483849   .0185634     2.61   0.010     .0118264    .0849434
         fsm |  -.0110086   .0028411    -3.87   0.000    -.0166038   -.0054135
         eal |   .0100141   .0013983     7.16   0.000     .0072603    .0127678
         sen |   -.001666   .0034806    -0.48   0.633    -.0085206    .0051886
    log_size |   .0078407   .0690823     0.11   0.910    -.1282089    .1438903
years_sinc~d |   .0020952   .0061385     0.34   0.733    -.0099939    .0141842
     academy |  -.0154287   .0579261    -0.27   0.790    -.1295075    .0986502
   urban_bin |


Linear regression                               Number of obs     =        267
                                                F(13, 253)        =      19.62
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4448
                                                Root MSE          =     .42434

------------------------------------------------------------------------------
             |             Robust HC3


p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
warmth_int~y |   .0225911   .0161137     1.40   0.162    -.0091429    .0543251
strict_int~y |     .07641   .0352194     2.17   0.031     .0070495    .1457706
         ks2 |   .0761886   .0192471     3.96   0.000     .0382837    .1140936
         fsm |  -.0143946   .0031248    -4.61   0.000    -.0205486   -.0082406
         eal |   .0130284   .0015661     8.32   0.000     .0099441    .0161127
         sen |  -.0025246   .0041052    -0.61   0.539    -.0106093      .00556
    log_size |  -.0121907    .085646    -0.14   0.887    -.1808606    .1564791
years_sinc~d |   .0052982   .0067325     0.79   0.432    -.0079607     .018557
     academy |   .0569966   .0620746     0.92   0.359    -.0652521    .1792453
   urban_bin |   -.185097   .0740226    -2.50   0.013     -.330876    -.039318
   selective |  -.2685854   .1758944    -1.53   0.12

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
warmth_int~y |    .019015   .0184663     1.03   0.304    -.0173523    .0553822
strict_int~y |   .0654095   .0367408     1.78   0.076    -.0069473    .1377664
         ks2 |   .0721908   .0194909     3.70   0.000     .0338057    .1105758
         fsm |  -.0084625   .0031748    -2.67   0.008     -.014715   -.0022101
         eal |   .0068268   .0014614     4.67   0.000     .0039486    .0097049
         sen |  -.0071734    .004512    -1.59   0.113    -.0160593    .0017125
    log_size |   .0366628   .0822416     0.45   0.656    -.1253025    .1986281
years_sinc~d |    .011854   .0068862     1.72   0.086    -.0017076    .0254155
     academy |    .021837   .0748198     0.29   0.771     -.125512     .169186
   urban_bin |

In [18]:
* ================================================================
* Enacted vs Espoused comparison: same tier-1 schools, visit-only vs interview-only
* More direct comparison — holds sample constant
* ================================================================
estimates clear

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))
    
    * Visit-only (enacted)
    regress `outcome' gs_warmth_visit gs_strictness_visit $controls if tier1, vce(hc3)
    estimates store enacted_`lbl'
    
    * Interview-only (espoused)
    regress `outcome' warmth_interview_only strict_interview_only $controls if tier1, vce(hc3)
    estimates store espoused102_`lbl'
    
    display "Enacted vs Espoused on the tier-1 schools — `lbl':"
    estimates restore enacted_`lbl'
    display "  Enacted:  β_W=" %6.3f _b[gs_warmth_visit] "  β_S=" %6.3f _b[gs_strictness_visit]
    estimates restore espoused102_`lbl'
    display "  Espoused: β_W=" %6.3f _b[warmth_interview_only] "  β_S=" %6.3f _b[strict_interview_only]
}

* The estimation sample is smaller than tier1 (the Ofsted-grade dummies drop the
* ungraded schools), so read the N off the stored estimates rather than typing it
* into the title -- a hard-coded 102 sat in this caption against an N row of 96.
estimates restore enacted_English
local nsamp = e(N)

* Export enacted vs espoused table (4 component outcomes — Overall excluded to keep table width)
esttab enacted_English espoused102_English enacted_Maths espoused102_Maths ///
       enacted_EBaC espoused102_EBaC enacted_Open espoused102_Open ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_enacted_espoused.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_warmth_visit gs_strictness_visit warmth_interview_only strict_interview_only) ///
    coeflabels(gs_warmth_visit "Warmth (visit, \$W_{12}\$)" ///
               gs_strictness_visit "Strictness (visit, \$S_{12}\$)" ///
               warmth_interview_only "Warmth (interview, \$W_3\$)" ///
               strict_interview_only "Strictness (interview, \$\bar{S}_{34}\$)") ///
    mtitles("Enact" "Espo" "Enact" "Espo" "Enact" "Espo" "Enact" "Espo") ///
    mgroups("English" "Maths" "EBaC" "Open", ///
            pattern(1 0 1 0 1 0 1 0) prefix(\multicolumn{2}{c}{) suffix(}) ///
            span erepeat(\cmidrule(lr){@span})) ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) nonumbers ///
    title("Enacted vs. espoused culture scores as predictors (\$N=`nsamp'\$)")

display "tab_enacted_espoused.tex written."


Linear regression                               Number of obs     =         96
                                                F(13, 82)         =      15.48
                                                Prob > F          =     0.0000
                                                R-squared         =     0.6501
                                                Root MSE          =     .30162



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1502921   .0447124     3.36   0.001     .0613448    .2392393
gs_strictn~t |   .1298203   .0468298     2.77   0.007     .0366608    .2229797
         ks2 |   .1100375   .0269014     4.09   0.000      .056522    .1635529
         fsm |  -.0054845   .0038657    -1.42   0.160    -.0131745    .0022056
         eal |   .0112759   .0017424     6.47   0.000     .0078097    .0147421
         sen |   .0070134   .0050614     1.39   0.170    -.0030554    .0170822
    log_size |  -.0385436   .1028519    -0.37   0.709    -.2431488    .1660615
years_sinc~d |  -.0103161    .010224    -1.01   0.316    -.0306549    .0100227
     academy |    .037192    .094407     0.39   0.695    -.1506135    .2249976
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
warmth_int~y |   .0108033   .0323113     0.33   0.739    -.0534741    .0750807
strict_int~y |   .0473194   .0514853     0.92   0.361    -.0551013      .14974
         ks2 |   .1041863   .0311887     3.34   0.001      .042142    .1662307
         fsm |  -.0049697   .0052654    -0.94   0.348    -.0154443    .0055049
         eal |   .0095142   .0022848     4.16   0.000     .0049691    .0140593
         sen |   .0059213   .0070859     0.84   0.406    -.0081748    .0200173
    log_size |   .0053806   .1318831     0.04   0.968    -.2569769    .2677381
years_sinc~d |  -.0098344    .013152    -0.75   0.457    -.0359979    .0163292
     academy |  -.0258699   .1244727    -0.21   0.836    -.2734856    .2217459
   urban_bin |

(results espoused102_Overall are active now)
 Espoused: β_W= 0.011 β_S= 0.047

Linear regression                               Number of obs     =         96
                                                F(12, 82)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.6382
                                                Root MSE          =     .32726



------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1765795    .043382     4.07   0.000     .0902788    .2628801
gs_strictn~t |   .0864345   .0493691     1.75   0.084    -.0117763    .1846454
         ks2 |   .1231733   .0299718     4.11   0.000     .0635499    .1827968
         fsm |   -.004286   .0044701    -0.96   0.340    -.0131784    .0046064
         eal |   .0140587    .002112     6.66   0.000     .0098573    .0182602
         sen |   .0065597   .0052383     1.25   0.214    -.0038609    .0169802
    log_size |  -.1292581   .1065153    -1.21   0.228    -.3411511    .0826348
years_sinc~d |  -.0084408   .0110925    -0.76   0.449    -.0305073    .0136257
     academy |   .0816477   .0992325     0.82   0.413    -.1157574    .2790528
   urban_bin |

...

(results espoused102_EBaC are active now)
 Espoused: β_W= 0.042 β_S= 0.055

Linear regression                               Number of obs     =         96
                                                F(13, 82)         =      12.44
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5653
                                                Root MSE          =     .35832



------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1457066   .0586771     2.48   0.015     .0289791    .2624341
gs_strictn~t |    .135027   .0646678     2.09   0.040     .0063821    .2636718
         ks2 |   .1207491   .0284083     4.25   0.000      .064236    .1772622
         fsm |  -.0041004   .0048353    -0.85   0.399    -.0137194    .0055186
         eal |   .0077636   .0027025     2.87   0.005     .0023874    .0131398
         sen |   .0035823   .0065785     0.54   0.588    -.0095044    .0166691
    log_size |   .0412988   .1112585     0.37   0.711    -.1800298    .2626273
years_sinc~d |  -.0124087   .0114994    -1.08   0.284    -.0352847    .0104674
     academy |   .0128434    .113271     0.11   0.910    -.2124888    .2381755
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
warmth_int~y |  -.0175683   .0376501    -0.47   0.642    -.0924662    .0573297
strict_int~y |   .0801406   .0575998     1.39   0.168    -.0344437     .194725
         ks2 |   .1162657   .0343011     3.39   0.001     .0480299    .1845014
         fsm |  -.0040085   .0060249    -0.67   0.508    -.0159939    .0079769
         eal |   .0061619   .0028724     2.15   0.035     .0004478    .0118759
         sen |   .0030391   .0076789     0.40   0.693    -.0122367     .018315
    log_size |   .0781777   .1382263     0.57   0.573    -.1967984    .3531539
years_sinc~d |  -.0130977   .0140319    -0.93   0.353    -.0410116    .0148162
     academy |  -.0655368    .140109    -0.47   0.641    -.3442582    .2131847
   urban_bin |

(results enacted_English are active now)


(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_enacted_espoused.tex)
tab_enacted_espoused.tex written.


In [19]:
* ================================================================
* Exploratory: Warmth and strictness GAP as predictor
* (enacted - espoused): does cultural coherence predict outcomes?
* ================================================================
gen warmth_gap  = gs_warmth_visit - warmth_interview_only
gen strict_gap  = gs_strictness_visit - strict_interview_only

display "Warmth gap (V - I) distribution:"
summarize warmth_gap if tier1

display "Strictness gap (V - I) distribution:"
summarize strict_gap if tier1

* Quick correlations with outcomes
display _newline "Correlation: warmth gap vs P8 outcomes:"
correlate warmth_gap strict_gap p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg if tier1

(3,229 missing values generated)
(3,229 missing values generated)
Warmth gap (V - I) distribution:

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
  warmth_gap |        103   -.5131715    1.478059      -3.78       3.67
Strictness gap (V - I) distribution:

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
  strict_gap |        103   -.0741748      1.1686      -2.43        3.4

Correlation: warmth gap vs P8 outcomes:
(obs=103)

             | warmth~p strict~p p8~g_avg p8meam~g p8~c_avg p8meao~g
-------------+------------------------------------------------------
  warmth_gap |   1.0000
  strict_gap |   0.2719   1.0000
p8meaeng_avg |   0.0510   0.1651   1.0000
p8meamat_avg |   0.0204   0.3000   0.8178   1.0000
p8meaebac_~g |   0.0195   0.2466   0.8982   0.8670   1.0000
p8meaopen_~g |   0.1020   0.19

In [20]:
* ================================================================
* Export: tab_continuity_robustness.tex
* Primary (n≈95) vs continuity-restricted (unchanged HT, n≈23)
* ================================================================

* Re-run primary for comparison (components only — small n makes Overall redundant)
estimates clear
foreach outcome in p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8meaeng_avg", "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open")))
    regress `outcome' $WS $controls if tier1, vce(hc3)
    estimates store primary2_`lbl'
    regress `outcome' $WS $controls_ngrade if tier1 & ht_changed==0, vce(hc3)
    estimates store cont2_`lbl'
}

esttab primary2_English cont2_English primary2_Maths cont2_Maths ///
       primary2_EBaC cont2_EBaC primary2_Open cont2_Open ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_continuity_robustness.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_warmth_visit gs_strictness_visit) ///
    coeflabels(gs_warmth_visit "Warmth (\$W\$)" gs_strictness_visit "Strictness (\$S\$)") ///
    mtitles("Full" "Cont" "Full" "Cont" "Full" "Cont" "Full" "Cont") ///
    mgroups("English" "Maths" "EBaC" "Open", ///
            pattern(1 0 1 0 1 0 1 0) prefix(\multicolumn{2}{c}{) suffix(})) ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) nonumbers ///
    title("Headteacher continuity robustness (Full sample vs. unchanged HT)")

display "tab_continuity_robustness.tex written."


Linear regression                               Number of obs     =         96
                                                F(12, 82)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.6382
                                                Root MSE          =     .32726



------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1765795    .043382     4.07   0.000     .0902788    .2628801
gs_strictn~t |   .0864345   .0493691     1.75   0.084    -.0117763    .1846454
         ks2 |   .1231733   .0299718     4.11   0.000     .0635499    .1827968
         fsm |   -.004286   .0044701    -0.96   0.340    -.0131784    .0046064
         eal |   .0140587    .002112     6.66   0.000     .0098573    .0182602
         sen |   .0065597   .0052383     1.25   0.214    -.0038609    .0169802
    log_size |  -.1292581   .1065153    -1.21   0.228    -.3411511    .0826348
years_sinc~d |  -.0084408   .0110925    -0.76   0.449    -.0305073    .0136257
     academy |   .0816477   .0992325     0.82   0.413    -.1157574    .2790528
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .0415879   .2358123     0.18   0.863    -.4774314    .5606072
gs_strictn~t |   .2322103   .1869002     1.24   0.240    -.1791543    .6435749
         ks2 |   .1197665   .0862955     1.39   0.193    -.0701685    .3097015
         fsm |   -.005108   .0137281    -0.37   0.717    -.0353233    .0251074
         eal |   .0210124   .0060708     3.46   0.005     .0076506    .0343741
         sen |   .0323701    .031317     1.03   0.323     -.036558    .1012983
    log_size |  -.3797139    .513358    -0.74   0.475    -1.509607    .7501793
years_sinc~d |  -.0158375    .029006    -0.55   0.596    -.0796793    .0480044
     academy |   .2320561   .3845041     0.60   0.558    -.6142318    1.078344
   urban_bin |


Linear regression                               Number of obs     =         96
                                                F(12, 82)         =          .
                                                Prob > F          =          .
                                                R-squared         =     0.5378
                                                Root MSE          =     .33346

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1176421   .0556669     2.11   0.038      .006903    .2283812


gs_strictn~t |   .1094054   .0520123     2.10   0.038     .0059364    .2128744
         ks2 |   .0759678   .0243386     3.12   0.002     .0275507    .1243849
         fsm |  -.0079994    .003861    -2.07   0.041    -.0156801   -.0003186
         eal |   .0114507   .0018001     6.36   0.000     .0078697    .0150318
         sen |   .0097373   .0053759     1.81   0.074     -.000957    .0204316
    log_size |   -.026307   .1182283    -0.22   0.824    -.2615007    .2088866
years_sinc~d |  -.0110893   .0109021    -1.02   0.312     -.032777    .0105984
     academy |   .0049594   .1051502     0.05   0.962    -.2042178    .2141366
   urban_bin |    .054027   .1004805     0.54   0.592    -.1458608    .2539148
   selective |  -.6004951   .3277703    -1.83   0.071    -1.252535    .0515445
             |
ofsted_~2019 |
          3  |  -.1970265   .1036164    -1.90   0.061    -.4031526    .0090996
          4  |  -.2285584    .127774    -1.79   0.077    -.4827416    .0256248
             |
       

...

note: selective omitted because of collinearity.

Linear regression                               Number of obs     =         22
                                                F(10, 11)         =       0.62
                                                Prob > F          =     0.7681
                                                R-squared         =     0.5185
                                                Root MSE          =     .41874

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   -.014625   .2635904    -0.06   0.957    -.5947836    .5655335
gs_strictn~t |   .1899877   .2817244     0.67   0.514    -.4300835    .8100588
         ks2 |   .0890681   .1295362     0.69   0.506    -.1960392    .3741755
         fsm |  -.0050971   .0179551    -0


Linear regression                               Number of obs     =         96
                                                F(13, 82)         =      12.44
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5653
                                                Root MSE          =     .35832

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |   .1457066   .0586771     2.48   0.015     .0289791    .2624341
gs_strictn~t |    .135027   .0646678     2.09   0.040     .0063821    .2636718
         ks2 |

   .1207491   .0284083     4.25   0.000      .064236    .1772622
         fsm |  -.0041004   .0048353    -0.85   0.399    -.0137194    .0055186
         eal |   .0077636   .0027025     2.87   0.005     .0023874    .0131398
         sen |   .0035823   .0065785     0.54   0.588    -.0095044    .0166691
    log_size |   .0412988   .1112585     0.37   0.711    -.1800298    .2626273
years_sinc~d |  -.0124087   .0114994    -1.08   0.284    -.0352847    .0104674
     academy |   .0128434    .113271     0.11   0.910    -.2124888    .2381755
   urban_bin |   .0538976   .1113472     0.48   0.630    -.1676074    .2754026
   selective |  -.8845146   .3331152    -2.66   0.010    -1.547187   -.2218423
             |
ofsted_~2019 |
          3  |  -.0936016   .1918903    -0.49   0.627    -.4753326    .2881294
          4  |  -.1502945   .3147568    -0.48   0.634     -.776446    .4758571
             |
       _cons |  -14.73507   3.236599    -4.55   0.000     -21.1737   -8.296449
---------------------

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |  -.0267599   .2264226    -0.12   0.908    -.5251126    .4715929
gs_strictn~t |   .2893663   .2723656     1.06   0.311    -.3101062    .8888389
         ks2 |   .0828353   .1308459     0.63   0.540    -.2051546    .3708251
         fsm |  -.0071823   .0183915    -0.39   0.704    -.0476618    .0332972
         eal |   .0172416    .006495     2.65   0.022     .0029462     .031537
         sen |   .0002125   .0378032     0.01   0.996    -.0829916    .0834167
    log_size |  -.6825382   .7946217    -0.86   0.409    -2.431489    1.066412
years_sinc~d |   .0016363   .0446156     0.04   0.971     -.096562    .0998345
     academy |   .1112775   .5963031     0.19   0.855    -1.201177    1.423732
   urban_bin |

(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_continuity_robustness.tex)
tab_continuity_robustness.tex written.


In [21]:
* ================================================================
* NATIONAL EXTENSION (E3): Ofsted LLM strictness -> P8
* N ~3,194 schools with ofsted_llmstrictnessscore, p8mea_avg, and controls
* ofsted_llmstrictnessscore on 1-5 scale; gs_strictness_visit on 0-10
* Warmth omitted: no valid national enacted warmth source
* No Ofsted grade control (score derived from the same Ofsted report)
* ================================================================

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))

    regress `outcome' ofsted_llmstrictnessscore $controls_ngrade, vce(hc3)
    estimates store nat_`lbl'
    display _newline "National (E3) -- `lbl' (n=" e(N) "):"
    display "  beta_S_ofsted = " %6.3f _b[ofsted_llmstrictnessscore] ///
            "  (se=" %6.3f _se[ofsted_llmstrictnessscore] ")" ///
            "  p=" %5.3f (2*ttail(e(df_r), abs(_b[ofsted_llmstrictnessscore]/_se[ofsted_llmstrictnessscore])))
    display "  R2 = " %6.4f e(r2)
}
display _newline "Note: Tier 1 visit-based beta_S=0.120 on 0-10 scale; multiply Ofsted beta by 2 for approx comparison."

* NOTE (5 Aug 2026): this cell no longer writes the LaTeX table.
* tables/tab_national_strictness.tex is built by thesis/make_national_strictness.py
* from tables/a7_estimates.csv, which thesis/a7_national_strictness.do produces.
* That table has TWO panels -- the published spec plus the pre-COVID Ofsted grade
* robustness check -- and Chapter 3 narrates both. The esttab call that used to
* live here wrote a single-panel table to the same path, so whichever ran last
* won; on 5 Aug the notebook silently destroyed Panel B. The regressions above
* stay, because they are the displayed check that Panel A still reproduces.
* To refresh the table: run a7_national_strictness.do, then make_national_strictness.py.
display "Panel A reproduced above; table itself is built by make_national_strictness.py."



Linear regression                               Number of obs     =      3,147
                                                F(10, 3136)       =     359.37
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5216
                                                Root MSE          =     .34944



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
ofste~sscore |    .165874   .0083584    19.85   0.000     .1494855    .1822625
         ks2 |    .035914   .0053197     6.75   0.000     .0254837    .0463444
         fsm |  -.0120768   .0008799   -13.72   0.000    -.0138021   -.0103515
         eal |   .0090783   .0005394    16.83   0.000     .0080207     .010136
         sen |  -.0019676   .0012787    -1.54   0.124    -.0044747    .0005396
    log_size |   .1217338   .0235942     5.16   0.000     .0754722    .1679954
years_sinc~d |   .0099332   .0016703     5.95   0.000     .0066582    .0132082
     academy |   .0347275   .0163934     2.12   0.034     .0025847    .0668703
   urban_bin |  -.0690464   .0236783    -2.92   0.004     -.115473   -.0226199
   selective |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaeng_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
ofste~sscore |   .1517487   .0089565    16.94   0.000     .1341875      .16931
         ks2 |   .0470158   .0056967     8.25   0.000     .0358461    .0581855
         fsm |  -.0096485   .0009539   -10.11   0.000    -.0115189   -.0077781
         eal |   .0101408   .0005934    17.09   0.000     .0089773    .0113044
         sen |  -.0034034   .0013674    -2.49   0.013    -.0060845   -.0007224
    log_size |   .0674776   .0254347     2.65   0.008     .0176072    .1173479
years_sinc~d |   .0102553   .0018593     5.52   0.000     .0066098    .0139009
     academy |   .0376776   .0178623     2.11   0.035     .0026546    .0727005
   urban_bin |  -.0509323   .0255506    -1.99   0.046      -.10103   -.0008347
   selective |


Linear regression                               Number of obs     =      3,147
                                                F(10, 3136)       =     253.12
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4706
                                                Root MSE          =     .35217

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
ofste~sscore |   .1424956   .0084609    16.84   0.000     .1259061    .1590851
         ks2 |   .0175439   .0052728     3.33   0.001     .0072054    .0278824
         fsm |  -.0142347   .0008182   -17.40   0.000    -.0158391   -.0126303
         eal |   .0092217   .0004853    19.00   0.000     .0082702    .0101732
         sen


Linear regression                               Number of obs     =      3,147
                                                F(10, 3136)       =     346.12
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4988
                                                Root MSE          =     .41226

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
ofste~sscore |    .171123   .0101093    16.93   0.000     .1513015    .1909445
         ks2 |     .03506   .0062406     5.62   0.000     .0228238    .0472961
         fsm |  -.0157145   .0011036   -14.24   0.000    -.0178784   -.0135506
         eal |

    .010846    .000682    15.90   0.000     .0095088    .0121833
         sen |  -.0013415   .0014977    -0.90   0.370     -.004278     .001595
    log_size |   .1134243   .0365917     3.10   0.002     .0416781    .1851705
years_sinc~d |   .0091014   .0020516     4.44   0.000     .0050787    .0131241
     academy |    .040841   .0198019     2.06   0.039     .0020149     .079667
   urban_bin |  -.0985925   .0307465    -3.21   0.001    -.1588779   -.0383071
   selective |   .0001575   .0532306     0.00   0.998    -.1042128    .1045278
       _cons |   -4.78888   .6844702    -7.00   0.000    -6.130935   -3.446825
------------------------------------------------------------------------------

National (E3) -- EBaC (n=3147):
 beta_S_ofsted =  0.171 (se= 0.010) p=0.000
 R2 = 0.4988

Linear regression                               Number of obs     =      3,147
                                                F(10, 3136)       =     238.95
                                                Prob >

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
ofste~sscore |   .1904732   .0097316    19.57   0.000     .1713922    .2095542
         ks2 |   .0397054   .0059976     6.62   0.000     .0279457    .0514651
         fsm |  -.0089938   .0010421    -8.63   0.000     -.011037   -.0069506
         eal |   .0065421   .0006727     9.73   0.000     .0052232     .007861
         sen |  -.0028301   .0015446    -1.83   0.067    -.0058586    .0001983
    log_size |   .1958808   .0245602     7.98   0.000     .1477251    .2440365
years_sinc~d |   .0134619   .0020975     6.42   0.000     .0093493    .0175746
     academy |   .0552807    .020076     2.75   0.006     .0159173    .0946441
   urban_bin |   -.050695   .0289915    -1.75   0.080    -.1075392    .0061493
   selective |

In [22]:
* ================================================================
* SCORE CONSTRUCTION SENSITIVITY (G2b + G3) -- RETIRED 5 Aug 2026
* ================================================================
* This cell compared four ways of weighting a visit/interview composite:
* visit-only, 60/40 current, 60/40 v1 (iq-adjusted S4), and 50/50.
*
* There is no composite any more, so there is no weighting to be sensitive to.
* The question the cell was asking has been dissolved rather than answered.
*
* Its successor -- does the ENACTED measure behave differently from the ESPOUSED
* one on the same schools -- is a real question and is already answered in cell 18,
* which holds the tier1 sample constant and exports tab_enacted_espoused.tex.
*
* PROSE: DONE 5 Aug 2026. The "Score construction sensitivity" subsection of
* thesis/chapters/03_paper2.tex (sec:p2_sensitivity) has been removed and folded
* into the enacted/espoused discussion as two paragraphs: the composite is
* withdrawn, the reason is the +0.243 / +0.190 / +0.184 agreement between the two
* rounds, and no coefficient moves because the headline models always read
* gs_*_visit. thesis/tables/tab_score_sensitivity.tex is deleted. Nothing in the
* thesis now cites either. STILL OPEN: chapter 2 (02_paper1.tex) discusses the
* 60/40 composite at length -- sec:p1_composite and the sec:p1_results
* prediction exercise -- and has NOT been touched.

display "Score-construction sensitivity retired: the composite was abolished 5 Aug 2026."

Score-construction sensitivity retired: the composite was abolished 5 Aug 2026.


In [23]:
* ================================================================
* SEMH MECHANISM TEST (G7)
* H: strict schools accumulate lower SEMH share than baseline predicts
* Consistent with behavioural sorting/exclusion mechanism
* Tier 1: gs_strictness_visit; National: ofsted_llmstrictnessscore
* semh_baseline_2016 and semh_current are raw pupil COUNTS -- divide by size for shares
* ================================================================

cap drop semh_share_baseline semh_share_current
gen semh_share_baseline = semh_baseline_2016 / size * 100
gen semh_share_current  = semh_current       / size * 100
label var semh_share_baseline "SEMH prevalence 2015-16 (% of roll)"
label var semh_share_current  "SEMH prevalence 2023-24 (% of roll)"

display _newline "SEMH share distributions:"
summarize semh_share_baseline semh_share_current

* Spec 1: Tier 1 -- strictness (gold-standard visit)
regress semh_share_current gs_strictness_visit semh_share_baseline $controls_ngrade if tier1, vce(hc3)
estimates store semh_t1_s
display _newline "SEMH mechanism -- Tier 1 strictness (n=" e(N) "):"
display "  beta_S = " %6.3f _b[gs_strictness_visit] ///
        "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_strictness_visit]/_se[gs_strictness_visit])))

* Spec 2: Tier 1 -- warmth (gold-standard visit)
regress semh_share_current gs_warmth_visit semh_share_baseline $controls_ngrade if tier1, vce(hc3)
estimates store semh_t1_w
display _newline "SEMH mechanism -- Tier 1 warmth (n=" e(N) "):"
display "  beta_W = " %6.3f _b[gs_warmth_visit] ///
        "  p=" %5.3f (2*ttail(e(df_r), abs(_b[gs_warmth_visit]/_se[gs_warmth_visit])))

* Spec 3: National -- Ofsted LLM strictness (large-N)
regress semh_share_current ofsted_llmstrictnessscore semh_share_baseline $controls_ngrade, vce(hc3)
estimates store semh_nat_s
display _newline "SEMH mechanism -- National Ofsted strictness (n=" e(N) "):"
display "  beta_S_ofsted = " %6.3f _b[ofsted_llmstrictnessscore] ///
        "  p=" %5.3f (2*ttail(e(df_r), abs(_b[ofsted_llmstrictnessscore]/_se[ofsted_llmstrictnessscore])))

esttab semh_t1_s semh_t1_w semh_nat_s ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_semh_mechanism.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(gs_strictness_visit gs_warmth_visit ofsted_llmstrictnessscore semh_share_baseline) ///
    coeflabels(gs_strictness_visit       "Strictness (\$S_{\text{visit}}\$)" ///
               gs_warmth_visit           "Warmth (\$W_{\text{visit}}\$)" ///
               ofsted_llmstrictnessscore "Strictness (Ofsted LLM, 1--5)" ///
               semh_share_baseline       "SEMH prevalence 2015--16 (\%)") ///
    mtitles("Tier 1 (S)" "Tier 1 (W)" "National (S)") nonumbers ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) ///
    title("SEMH mechanism: culture and current SEMH composition conditional on baseline")

display "tab_semh_mechanism.tex written."


(325 missing values generated)
(7 missing values generated)

SEMH share distributions:

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
semh_share~e |      3,007    2.407793    2.567472          0    64.7541
semh_share~t |      3,325    4.146441    2.559118          0   32.72727

Linear regression                               Number of obs     =         95
                                                F(11, 83)         =       6.69
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5535
                                                Root MSE          =     1.4701



------------------------------------------------------------------------------
             |             Robust HC3
semh_share~t | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_strictn~t |  -.2906782   .3244908    -0.90   0.373    -.9360773     .354721
semh_share~e |   .1629647   .0670825     2.43   0.017     .0295403    .2963891
         ks2 |  -.1260831   .1135561    -1.11   0.270    -.3519415    .0997754
         fsm |   .0307018   .0210822     1.46   0.149    -.0112299    .0726336
         eal |  -.0317494   .0088426    -3.59   0.001    -.0493369   -.0141619
         sen |    .114325   .0405856     2.82   0.006     .0336019     .195048
    log_size |    -1.2327   .8025795    -1.54   0.128    -2.828998    .3635986
years_sinc~d |   .0162348   .0454257     0.36   0.722    -.0741151    .1065848
     academy |  -.1488438   .4904262    -0.30   0.762    -1.124282    .8265942
   urban_bin |


Linear regression                               Number of obs     =         95
                                                F(11, 83)         =       6.93
                                                Prob > F          =     0.0000
                                                R-squared         =     0.5415
                                                Root MSE          =     1.4895

------------------------------------------------------------------------------
             |             Robust HC3
semh_share~t | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
gs_warmth_~t |  -.0933672   .2407851    -0.39   0.699     -.572279    .3855447
semh_share~e |   .1786872   .0629501     2.84   0.006      .053482    .3038924
         ks2 |  -.1106044   .1127162    -0.98   0.329    -.3347924    .1135837
         fsm |   .0319871   .0213246     1.50   0.137    -.0104266    .0744009
         eal


Linear regression                               Number of obs     =      2,909
                                                F(11, 2897)       =     150.48
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4897
                                                Root MSE          =      1.761



------------------------------------------------------------------------------
             |             Robust HC3
semh_share~t | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
ofste~sscore |  -.1655615   .0431872    -3.83   0.000    -.2502422   -.0808808
semh_share~e |   .1848295   .0218258     8.47   0.000     .1420338    .2276253
         ks2 |  -.0664424   .0228499    -2.91   0.004     -.111246   -.0216387
         fsm |   .0215296   .0038304     5.62   0.000     .0140189    .0290403
         eal |   -.022436   .0019835   -11.31   0.000    -.0263252   -.0185469
         sen |   .1609919   .0105692    15.23   0.000     .1402679    .1817159
    log_size |  -.5913958   .1534863    -3.85   0.000    -.8923491   -.2904426
years_sinc~d |   -.010843   .0078975    -1.37   0.170    -.0263284    .0046424
     academy |   .0744323   .0839058     0.89   0.375    -.0900888    .2389534
   urban_bin |

(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_semh_mechanism.tex)
tab_semh_mechanism.tex written.


In [24]:
* ================================================================
* MANAGEMENT DISCOURSE (G4): trx_management -> P8
* REPOINTED 5 Aug 2026 onto the segmented scorers. The results previously stored in
* this notebook (beta_M about -0.09, all p > 0.17) were produced by the v6 bundled
* scorer, which is dead; they do not carry over and must not be quoted.
*
* RE-RUN COMPLETE (5 Aug 2026), n=282. Segmented-scorer results:
*   Overall  beta_W=+0.022  beta_S=+0.012  beta_M=-0.023  p_M=0.250
* The management-discourse coefficient is null on every outcome. The substantive
* conclusion previously drawn from the v6 run is WITHDRAWN: v6 collapsed (85% of
* schools scored exactly 4, SD 0.38), so it could not have supported a conclusion
* either way. The segmented scorer has usable dispersion (mean 3.12, SD 1.26,
* 28% modal) and still finds nothing, which is the informative version of the null.
* Extended tier: n ~287 schools with interview transcript LLM scores + P8 + controls
* All trx_llm scores on 1-5 scale (not 0-10 like visit-based gold standard)
* Filter directly on !missing(trx_management) -- avoids redefining tier2
* ================================================================

count if !missing(trx_management) & !missing(p8mea_avg) & !missing(ks2)
display _newline "Sample: schools with trx_management above"

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))

    regress `outcome' trx_warmth trx_strictness trx_management ///
        $controls_ngrade if !missing(trx_management), vce(hc3)
    estimates store mgmt_`lbl'
    display _newline "Management discourse -- `lbl' (n=" e(N) "):"
    display "  beta_W(trx)=" %6.3f _b[trx_warmth] ///
            "  beta_S(trx)=" %6.3f _b[trx_strictness] ///
            "  beta_M(trx)=" %6.3f _b[trx_management] ///
            "  p_M=" %5.3f (2*ttail(e(df_r), abs(_b[trx_management]/_se[trx_management])))
}

* Read N off the stored estimates: this caption carried a hard-coded 282.
estimates restore mgmt_Overall
local nmg = e(N)

esttab mgmt_Overall mgmt_English mgmt_Maths mgmt_EBaC mgmt_Open ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_management_discourse.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(trx_warmth trx_strictness trx_management) ///
    coeflabels(trx_warmth      "Warmth (transcript LLM, $\tilde{W}$)" ///
               trx_strictness  "Strictness (transcript LLM, $\tilde{S}$)" ///
               trx_management  "Management discourse (transcript LLM, $\tilde{M}$)") ///
    mlabels("Overall" "English" "Maths" "EBaC" "Open") nonumbers ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) ///
    title("Management discourse as predictor: extended tier (\$N = `nmg'\$)")
* POST-RUN: esttab emits neither \label nor \resizebox, and Chapter 3 \cref{}s
* this table. Do NOT re-add them by hand -- the next re-run strips them again.
* Run  python thesis/fix_esttab_tables.py  after this notebook instead.

display "tab_management_discourse.tex written."

  287

Sample: schools with trx_management above

Linear regression                               Number of obs     =        282
                                                F(12, 269)        =      20.08
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4267
                                                Root MSE          =     .37805



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
  trx_warmth |   .0211888   .0179218     1.18   0.238    -.0140961    .0564737
trx_strict~s |   .0123883    .019901     0.62   0.534    -.0267933    .0515698
trx_manage~t |  -.0221684   .0201139    -1.10   0.271     -.061769    .0174322
         ks2 |   .0653085    .016578     3.94   0.000     .0326694    .0979476
         fsm |  -.0117312   .0028278    -4.15   0.000    -.0172986   -.0061639
         eal |   .0100527   .0012923     7.78   0.000     .0075084    .0125969
         sen |  -.0038729   .0033936    -1.14   0.255    -.0105544    .0028086
    log_size |    .054819   .0651214     0.84   0.401    -.0733935    .1830315
years_sinc~d |   .0090949    .005558     1.64   0.103    -.0018477    .0200375
     academy |


Linear regression                               Number of obs     =        282
                                                F(12, 269)        =      13.63
                                                Prob > F          =     0.0000
                                                R-squared         =     0.3648
                                                Root MSE          =     .38095

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
  trx_warmth |  -.0002533   .0176506    -0.01   0.989    -.0350042    .0344976
trx_strict~s |   .0094019   .0196679     0.48   0.633    -.0293207    .0481245
trx_manage~t |  -.0262128   .0197095    -1.33   0.185    -.0650173    .0125918
         ks2 |   .0510293   .0170667     2.99   0.003      .017428    .0846305
         fsm


Linear regression                               Number of obs     =        282
                                                F(12, 269)        =      20.14
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4442
                                                Root MSE          =     .43342

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
  trx_warmth |   .0238349   .0206651     1.15   0.250    -.0168511    .0645208
trx_strict~s |   .0323768   .0214906     1.51   0.133    -.0099344    .0746879
trx_manage~t |  -.0235196   .0234082    -1.00   0.316    -.0696062     .022567
         ks2 |   .0754261   .0185395     4.07   0.000     .0389251    .1119271
         fsm


Linear regression                               Number of obs     =        282
                                                F(12, 269)        =      13.78
                                                Prob > F          =     0.0000
                                                R-squared         =     0.3096
                                                Root MSE          =     .47533

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
  trx_warmth |   .0291645    .022385     1.30   0.194    -.0149076    .0732365
trx_strict~s |  -.0034842   .0262058    -0.13   0.894    -.0550787    .0481104
trx_manage~t |  -.0196224   .0240114    -0.82   0.415    -.0668967    .0276518
         ks2 |   .0655623    .020308     3.23   0.001     .0255795    .1055452
         fsm


Management discourse -- Open (n=282):
 beta_W(trx)= 0.029 beta_S(trx)=-0.003 beta_M(trx)=-0.020 p_M=0.415
(results mgmt_Overall are active now)


(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_management_discourse.tex)
tab_management_discourse.tex written.


In [25]:
* ================================================================
* TEACHING PHILOSOPHY (national): web_id_llmteachingphilosophy -> P8
* 3 categories (v3 rubric): traditional (111), progressive (34), unmarked base (3,166)
* web_id_llmteachingphilosophy is a string variable (lowercase after case(lower))
* ================================================================

cap drop trad prog
gen trad = (web_id_llmteachingphilosophy == "traditional") if !missing(web_id_llmteachingphilosophy)
gen prog = (web_id_llmteachingphilosophy == "progressive") if !missing(web_id_llmteachingphilosophy)
label var trad "Traditional teaching philosophy (vs unmarked)"
label var prog "Progressive teaching philosophy (vs unmarked)"

display _newline "Category counts:"
count if trad == 1
count if prog == 1
count if !missing(web_id_llmteachingphilosophy) & !missing(p8mea_avg) & !missing(ks2)

foreach outcome in p8mea_avg p8meaeng_avg p8meamat_avg p8meaebac_avg p8meaopen_avg {
    local lbl = cond("`outcome'"=="p8mea_avg",    "Overall", ///
                cond("`outcome'"=="p8meaeng_avg",  "English", ///
                cond("`outcome'"=="p8meamat_avg",  "Maths",   ///
                cond("`outcome'"=="p8meaebac_avg", "EBaC",    "Open"))))

    regress `outcome' trad prog $controls_ngrade, vce(hc3)
    estimates store tp_`lbl'
    display _newline "Teaching philosophy -- `lbl' (n=" e(N) "):"
    display "  beta_trad=" %6.3f _b[trad] "  p=" %5.3f (2*ttail(e(df_r), abs(_b[trad]/_se[trad])))
    display "  beta_prog=" %6.3f _b[prog] "  p=" %5.3f (2*ttail(e(df_r), abs(_b[prog]/_se[prog])))
}

* Same reason as cell 18: the caption said 3,219 against an N row of 3,170.
local ntp = subinstr(string(e(N), "%9.0fc"), ",", "{,}", .)

esttab tp_Overall tp_English tp_Maths tp_EBaC tp_Open ///
    using "C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tables/tab_teaching_philosophy.tex", ///
    replace booktabs label se star(* 0.10 ** 0.05 *** 0.01) ///
    keep(trad prog) ///
    coeflabels(trad "Traditional (vs unmarked)" prog "Progressive (vs unmarked)") ///
    mlabels("Overall" "English" "Maths" "EBaC" "Open") nonumbers ///
    stats(N r2, fmt(%9.0f %8.3f)) b(%8.3f) se(%8.3f) ///
    title("Website teaching philosophy and Progress~8 (national, \$N = `ntp'\$)")

display "tab_teaching_philosophy.tex written."


(21 missing values generated)
(21 missing values generated)

Category counts:
  111
  34
  3,219

Linear regression                               Number of obs     =      3,170
                                                F(11, 3158)       =     232.74
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4592
                                                Root MSE          =     .37294



------------------------------------------------------------------------------
             |             Robust HC3
   p8mea_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        trad |   .1239217   .0453663     2.73   0.006     .0349713    .2128722
        prog |  -.0876527   .0714914    -1.23   0.220     -.227827    .0525216
         ks2 |   .0504065   .0055903     9.02   0.000     .0394455    .0613676
         fsm |  -.0135386   .0009169   -14.77   0.000    -.0153364   -.0117408
         eal |   .0106898   .0005564    19.21   0.000      .009599    .0117807
         sen |  -.0018399   .0013462    -1.37   0.172    -.0044794    .0007997
    log_size |   .1307249   .0226136     5.78   0.000      .086386    .1750638
years_sinc~d |   .0117993     .00178     6.63   0.000     .0083092    .0152893
     academy |   .0217086   .0172799     1.26   0.209    -.0121723    .0555895
   urban_bin |


Linear regression                               Number of obs     =      3,170
                                                F(11, 3158)       =     174.60
                                                Prob > F          =     0.0000
                                                R-squared         =     0.4214
                                                Root MSE          =     .36931

------------------------------------------------------------------------------
             |             Robust HC3
p8meamat_avg | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        trad |    .114044   .0444562     2.57   0.010     .0268781      .20121
        prog |  -.0236513   .0614489    -0.38   0.700    -.1441351    .0968324
         ks2 |   .0298282   .0054423     5.48   0.000     .0191575    .0404989
         fsm |  -.0155234   .0008495   -18.27   0.000    -.0171889   -.0138578


         eal |   .0105845   .0005002    21.16   0.000     .0096037    .0115653
         sen |  -.0010921   .0012052    -0.91   0.365    -.0034551    .0012709
    log_size |    .092791   .0195296     4.75   0.000     .0544991    .1310829
years_sinc~d |   .0081007   .0017486     4.63   0.000     .0046723    .0115292
     academy |  -.0148999   .0166053    -0.90   0.370    -.0474582    .0176585
   urban_bin |  -.0608784    .022386    -2.72   0.007    -.1047711   -.0169858
   selective |  -.0009308   .0494809    -0.02   0.985    -.0979487    .0960871
       _cons |  -3.460073   .5824252    -5.94   0.000    -4.602043   -2.318103
------------------------------------------------------------------------------

Teaching philosophy -- Maths (n=3170):
 beta_trad= 0.114 p=0.010
 beta_prog=-0.024 p=0.700

Linear regression                               Number of obs     =      3,170
                                                F(11, 3158)       =     230.55
                                      

------------------------------------------------------------------------------
             |             Robust HC3
p8meaebac_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        trad |   .1457914   .0522256     2.79   0.005     .0433919     .248191
        prog |  -.0539613   .0912677    -0.59   0.554    -.2329113    .1249886
         ks2 |   .0497555   .0064299     7.74   0.000     .0371483    .0623627
         fsm |  -.0172549   .0011224   -15.37   0.000    -.0194556   -.0150542
         eal |    .012544   .0006874    18.25   0.000     .0111962    .0138919
         sen |  -.0012155   .0015627    -0.78   0.437    -.0042794    .0018485
    log_size |   .1228795   .0348644     3.52   0.000     .0545204    .1912386
years_sinc~d |   .0111442   .0021536     5.17   0.000     .0069216    .0153668
     academy |    .028019   .0205035     1.37   0.172    -.0121825    .0682205
   urban_bin |

------------------------------------------------------------------------------
             |             Robust HC3
p8meaopen_~g | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        trad |   .1156436   .0506438     2.28   0.022     .0163456    .2149416
        prog |   -.188408   .1002455    -1.88   0.060    -.3849609    .0081448
         ks2 |   .0567125   .0063413     8.94   0.000     .0442791     .069146
         fsm |  -.0105999    .001087    -9.75   0.000    -.0127313   -.0084685
         eal |   .0083672    .000688    12.16   0.000     .0070181    .0097162
         sen |   -.002671   .0016067    -1.66   0.097    -.0058213    .0004793
    log_size |    .203837   .0246031     8.29   0.000     .1555972    .2520767
years_sinc~d |   .0155456   .0022065     7.05   0.000     .0112193    .0198719
     academy |    .039277   .0208833     1.88   0.060    -.0016691    .0802232
   urban_bin |

(output written to C:/Users/damia/OneDrive/Documents/Schools Project/thesis/tab
> les/tab_teaching_philosophy.tex)
tab_teaching_philosophy.tex written.
